# Run Massey Pipeline
Execute the script to compute HCA-adjusted historical Massey ratings, validate vs official MAS, and generate momentum features.

In [2]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

project_root = Path.cwd().parent
source_path = project_root / "data" / "source"
processed_path = project_root / "data" / "processed"

name_matching_path = processed_path / "name_matching"
modeling_path = processed_path / "modeling_datasets"
modeling_diag_path = modeling_path / "diagnostics"
massey_path = processed_path / "massey"

for p in [processed_path, name_matching_path, modeling_path, modeling_diag_path, massey_path]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Project root        : {project_root}")
print(f"Source path         : {source_path}")
print(f"Processed path      : {processed_path}")
print(f"Name matching path  : {name_matching_path}")
print(f"Modeling path       : {modeling_path}")
print(f"Modeling diag path  : {modeling_diag_path}")
print(f"Massey path         : {massey_path}")

Project root        : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM
Source path         : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\source
Processed path      : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed
Name matching path  : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\name_matching
Modeling path       : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets
Modeling diag path  : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics
Massey path         : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\massey


In [3]:
# Massey pipeline run is intentionally deferred.
# Run the dedicated Massey cell right after ESPN data gathering (next section).
script_path = project_root / "massey_pipeline.py"
print("Massey pipeline deferred until after ESPN ingestion/refresh.")
print(f"Script path: {script_path}")

Massey pipeline deferred until after ESPN ingestion/refresh.
Script path: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\massey_pipeline.py


In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------------------------------------
# Dual-tournament pipeline config and ingestion utilities
# -------------------------------------------------------------
TOURNAMENT_TYPES = ("M", "W")

TOURNAMENT_CONFIG = {
    "M": {
        "prefix": "M",
        # Source files are .parquet on GitHub
        "remote_box_url": "https://raw.githubusercontent.com/sportsdataverse/hoopR-mbb-data/main/mbb/team_box/parquet/team_box_{season}.parquet",
        "massey_file": "m_rating_momentum_features.csv",
        "seeding_sheet": "MNCAATourneySeeds",
        "local_path": source_path / "M_combined_team_box_scores.csv"
    },
    "W": {
        "prefix": "W",
        # Source files are .parquet on GitHub
        "remote_box_url": "https://raw.githubusercontent.com/sportsdataverse/wehoop-wbb-data/main/wbb/team_box/parquet/team_box_{season}.parquet",
        "massey_file": "w_rating_momentum_features.csv",
        "seeding_sheet": "WNCAATourneySeeds",
        "local_path": source_path / "W_combined_team_box_scores.csv"
    },
}

CLUB_SHARE_EXPORT_NAMES = {
    "M": {
        "historical": "m_historical_matchups_2005_2025.csv",
        "all_possible_2026": "m_modeling_matchups_2026_all_possible.csv",
        "team_aggregates": "m_team_aggregates_2005_2026.csv",
    },
    "W": {
        "historical": "w_historical_matchups_2014_2025.csv",
        "all_possible_2026": "w_modeling_matchups_2026_all_possible.csv",
        "team_aggregates": "w_team_aggregates_2014_2026.csv",
    },
}

CLUB_SHARE_PATH = processed_path / "club_share"
CLUB_SHARE_PATH.mkdir(parents=True, exist_ok=True)

selection_sunday_path = source_path / "selection_sunday_dates.csv"
mens_boxscore_csv_path = TOURNAMENT_CONFIG["M"]["local_path"]
womens_boxscore_csv_path = TOURNAMENT_CONFIG["W"]["local_path"]

season_start_m, season_start_w, season_end = 2005, 2014, 2026
refresh_season = 2026


def fetch_season_schedule(tournament_type: str, season: int) -> pd.DataFrame:
    """Fetches remote .parquet source and returns a DataFrame."""
    url = TOURNAMENT_CONFIG[tournament_type]["remote_box_url"].format(season=season)
    # Using read_parquet because source files are Parquet
    season_df = pd.read_parquet(url)
    if "season" not in season_df.columns:
        season_df["season"] = season
    return season_df


def harmonize_refresh_to_local(local_df: pd.DataFrame, refresh_df: pd.DataFrame) -> pd.DataFrame:
    aligned = refresh_df.copy()
    for col in local_df.columns:
        if col not in aligned.columns:
            aligned[col] = pd.NA
    aligned = aligned[local_df.columns]
    for col in local_df.columns:
        local_dtype = local_df[col].dtype
        if pd.api.types.is_integer_dtype(local_dtype) or pd.api.types.is_float_dtype(local_dtype):
            aligned[col] = pd.to_numeric(aligned[col], errors="coerce")
        elif pd.api.types.is_datetime64_any_dtype(local_dtype):
            aligned[col] = pd.to_datetime(aligned[col], errors="coerce")
        else:
            aligned[col] = aligned[col].where(aligned[col].isna(), aligned[col].astype(str))
    return aligned


def write_canonical_csv(df_to_save: pd.DataFrame, out_path: Path) -> None:
    """Saves DataFrame as a CSV."""
    temp_path = out_path.with_suffix(".tmp.csv")
    df_to_save.to_csv(temp_path, index=False)
    temp_path.replace(out_path)


def load_historical_boxscores(tournament_type: str, start_season: int, end_season: int) -> pd.DataFrame:
    print(f"[{tournament_type}] No local CSV found. Performing full remote Parquet rebuild ({start_season}-{end_season})...")
    season_frames = []
    for season in range(start_season, end_season + 1):
        try:
            season_frames.append(fetch_season_schedule(tournament_type, season))
        except Exception as exc:
            print(f"Warning: could not fetch {tournament_type} season {season}: {exc}")
    
    if not season_frames:
        raise RuntimeError(f"No {tournament_type} seasons could be fetched from remote.")
    
    full_df = pd.concat(season_frames, ignore_index=True)
    write_canonical_csv(full_df, TOURNAMENT_CONFIG[tournament_type]["local_path"])
    return full_df


def load_boxscores_with_refresh(tournament_type: str, start_season: int, end_season: int, refresh_season: int) -> pd.DataFrame:
    local_path = TOURNAMENT_CONFIG[tournament_type]["local_path"]
    
    if not local_path.exists():
        return load_historical_boxscores(tournament_type, start_season, end_season)

    try:
        # Local file is CSV
        local_df = pd.read_csv(local_path, low_memory=False)
        print(f"Loaded local {tournament_type} CSV: {local_path} ({len(local_df):,} rows)")
    except Exception as exc:
        print(f"Warning: local {tournament_type} CSV unreadable ({exc}). Attempting remote rebuild.")
        return load_historical_boxscores(tournament_type, start_season, end_season)

    try:
        # Remote refresh is Parquet
        refresh_df = fetch_season_schedule(tournament_type, refresh_season)
        refresh_df = harmonize_refresh_to_local(local_df, refresh_df)

        season_mask = pd.to_numeric(local_df["season"], errors="coerce").fillna(-1)
        base_df = local_df[season_mask != refresh_season].copy()
        merged_df = pd.concat([base_df, refresh_df], ignore_index=True)

        write_canonical_csv(merged_df, local_path)
        print(f"Refreshed {tournament_type} season {refresh_season}. Local CSV rows now: {len(merged_df):,}")
        return merged_df
    except Exception as exc:
        print(f"Warning: could not refresh {tournament_type} season {refresh_season} ({exc}). Using local CSV data.")
        return local_df


def _coalesce_columns(frame: pd.DataFrame, candidates: list[str], default=np.nan) -> pd.Series:
    out = pd.Series(default, index=frame.index)
    for col in candidates:
        if col in frame.columns:
            out = out.where(out.notna(), frame[col])
    return out


def _safe_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _normalize_home_away(value: object) -> str:
    if pd.isna(value): return "N"
    raw = str(value).strip().lower()
    if raw in {"h", "home", "vs", "v", "1"}: return "H"
    if raw in {"a", "away", "@", "-1"}: return "A"
    return "N"


def normalize_team_box_schema(raw_df: pd.DataFrame, tournament_type: str) -> pd.DataFrame:
    canonical = pd.DataFrame(index=raw_df.index)
    canonical["season"] = _safe_numeric(_coalesce_columns(raw_df, ["season", "Season"]))
    canonical["game_id"] = _safe_numeric(_coalesce_columns(raw_df, ["game_id", "GameID", "game_pk", "id"]))
    canonical["game_date"] = pd.to_datetime(_coalesce_columns(raw_df, ["game_date", "date", "game_datetime", "start_date"]), errors="coerce")
    canonical["team_id"] = _safe_numeric(_coalesce_columns(raw_df, ["team_id", "teamid", "TeamID"]))
    canonical["opponent_team_id"] = _safe_numeric(_coalesce_columns(raw_df, ["opponent_team_id", "opponent_id", "opponent_teamid", "OppTeamID"]))
    canonical["team_score"] = _safe_numeric(_coalesce_columns(raw_df, ["team_score", "points", "score", "team_points"]))
    canonical["opponent_score"] = _safe_numeric(_coalesce_columns(raw_df, ["opponent_team_score", "opponent_score", "opponent_points", "opp_score", "opp_points"]))
    canonical["field_goals_made"] = _safe_numeric(_coalesce_columns(raw_df, ["field_goals_made", "fgm", "fg_made"]))
    canonical["field_goals_attempted"] = _safe_numeric(_coalesce_columns(raw_df, ["field_goals_attempted", "fga", "fg_attempted"]))
    canonical["three_point_field_goals_made"] = _safe_numeric(_coalesce_columns(raw_df, ["three_point_field_goals_made", "fg3m", "three_pt_made"]))
    canonical["three_point_field_goals_attempted"] = _safe_numeric(_coalesce_columns(raw_df, ["three_point_field_goals_attempted", "fg3a", "three_pt_attempted"]))
    canonical["free_throws_made"] = _safe_numeric(_coalesce_columns(raw_df, ["free_throws_made", "ftm", "free_throw_made"]))
    canonical["free_throws_attempted"] = _safe_numeric(_coalesce_columns(raw_df, ["free_throws_attempted", "fta", "free_throw_attempted"]))
    canonical["offensive_rebounds"] = _safe_numeric(_coalesce_columns(raw_df, ["offensive_rebounds", "orb", "oreb"]))
    canonical["defensive_rebounds"] = _safe_numeric(_coalesce_columns(raw_df, ["defensive_rebounds", "drb", "dreb"]))
    canonical["total_rebounds"] = _safe_numeric(_coalesce_columns(raw_df, ["total_rebounds", "reb", "rebounds"]))
    canonical["turnovers"] = _safe_numeric(_coalesce_columns(raw_df, ["turnovers", "tov"]))
    canonical["assists"] = _safe_numeric(_coalesce_columns(raw_df, ["assists", "ast"]))
    canonical["steals"] = _safe_numeric(_coalesce_columns(raw_df, ["steals", "stl"]))
    canonical["blocks"] = _safe_numeric(_coalesce_columns(raw_df, ["blocks", "blk"]))
    canonical["personal_fouls"] = _safe_numeric(_coalesce_columns(raw_df, ["personal_fouls", "fouls", "team_fouls", "pf"]))
    canonical["team_home_away"] = _coalesce_columns(raw_df, ["team_home_away", "home_away", "location", "game_location"]).map(_normalize_home_away)
    canonical["team_name"] = _coalesce_columns(raw_df, ["team_name", "team_display_name", "display_name", "team_short_display_name", "short_display_name", "school"], default=pd.NA)
    canonical["opponent_team_name"] = _coalesce_columns(raw_df, ["opponent_team_name", "opponent_team_display_name", "opponent_display_name", "opponent_team_short_display_name", "opponent_short_display_name", "opponent_school"], default=pd.NA)
    canonical["team_location"] = _coalesce_columns(raw_df, ["team_location", "location_name", "team_city"], default=pd.NA)
    canonical["team_display_name"] = _coalesce_columns(raw_df, ["team_display_name", "display_name"], default=pd.NA)
    canonical["team_short_display_name"] = _coalesce_columns(raw_df, ["team_short_display_name", "short_display_name"], default=pd.NA)
    canonical["team_abbreviation"] = _coalesce_columns(raw_df, ["team_abbreviation", "abbreviation", "team_abbr"], default=pd.NA)
    canonical["team_color"] = _coalesce_columns(raw_df, ["team_color"], default=pd.NA)
    canonical["team_alternate_color"] = _coalesce_columns(raw_df, ["team_alternate_color"], default=pd.NA)
    canonical["team_logo"] = _coalesce_columns(raw_df, ["team_logo"], default=pd.NA)
    canonical["tournament_type"] = tournament_type

    s_start = season_start_m if tournament_type == "M" else season_start_w
    canonical = canonical[canonical["season"].between(s_start, season_end)].copy()
    canonical = canonical[canonical["game_date"].notna()].copy()

    required = ["team_id", "opponent_team_id", "team_score", "field_goals_attempted", "turnovers"]
    missing_mask = canonical[required].isna().any(axis=1)
    if missing_mask.any():
        canonical = canonical[~missing_mask].copy()

    return canonical.reset_index(drop=True)


def parse_selection_sunday_dates(path: Path) -> pd.DataFrame:
    if not path.exists(): return pd.DataFrame(columns=["Season", "selection_sunday_date"])
    selection_df = pd.read_csv(path)
    if "Season" not in selection_df.columns or "Selection Sunday Date" not in selection_df.columns:
        return pd.DataFrame(columns=["Season", "selection_sunday_date"])
    selection_df["Season"] = pd.to_numeric(selection_df["Season"], errors="coerce")
    selection_df["Selection Sunday Date"] = selection_df["Selection Sunday Date"].astype(str).str.replace("*", "", regex=False).str.strip()
    selection_df["selection_sunday_date"] = pd.to_datetime(selection_df["Selection Sunday Date"], errors="coerce")
    selection_df = selection_df.dropna(subset=["Season", "selection_sunday_date"])
    selection_df["Season"] = selection_df["Season"].astype(int)
    return selection_df[["Season", "selection_sunday_date"]].drop_duplicates("Season")


def apply_selection_sunday_cutoff(frame: pd.DataFrame, selection_df: pd.DataFrame) -> pd.DataFrame:
    if selection_df.empty: return frame.copy()
    out = frame.copy()
    out["season"] = _safe_numeric(out["season"]).astype("Int64")
    out = out.merge(selection_df, left_on="season", right_on="Season", how="left")
    has_cutoff = out["selection_sunday_date"].notna()
    out = out[(~has_cutoff) | (out["game_date"] < out["selection_sunday_date"])].copy()
    return out.drop(columns=["Season", "selection_sunday_date"], errors="ignore")


def _pipe_join(values: pd.Series) -> str:
    uniq = sorted({str(v).strip() for v in values.dropna() if str(v).strip()})
    return "|".join(uniq)


def build_team_summary_from_boxscores(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.groupby("team_id", as_index=False).agg(
        team_name=("team_name", _pipe_join),
        team_location=("team_location", _pipe_join),
        team_display_name=("team_display_name", _pipe_join),
        team_short_display_name=("team_short_display_name", _pipe_join),
        team_abbreviation=("team_abbreviation", _pipe_join),
        team_color=("team_color", _pipe_join),
        team_alternate_color=("team_alternate_color", _pipe_join),
        team_logo=("team_logo", _pipe_join),
        seasons_covered=("season", lambda s: _pipe_join(pd.Series(sorted(set(s.dropna().astype(int).tolist()))))),
    )

# ----------------------------- Execution ---------------------------------------
selection_sunday_df = parse_selection_sunday_dates(selection_sunday_path)
raw_boxscores, normalized_boxscores, team_summaries = {}, {}, {}

for t_type in TOURNAMENT_TYPES:
    s_start = season_start_m if t_type == "M" else season_start_w
    # This will load from local CSV if it exists, or fetch remote Parquet if not.
    raw_df = load_boxscores_with_refresh(t_type, s_start, season_end, refresh_season)
    
    norm_df = normalize_team_box_schema(raw_df, t_type)
    norm_df = apply_selection_sunday_cutoff(norm_df, selection_sunday_df)
    
    summary_df = build_team_summary_from_boxscores(norm_df)
    summary_out_path = source_path / f"{t_type}_combined_team_box_scores_team_summary.csv"
    summary_df.to_csv(summary_out_path, index=False)

    raw_boxscores[t_type], normalized_boxscores[t_type], team_summaries[t_type] = raw_df, norm_df, summary_df
    print(f"[{t_type}] Final: {len(norm_df):,} rows, {summary_df['team_id'].nunique():,} teams.")

Loaded local M CSV: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\source\M_combined_team_box_scores.csv (250,568 rows)
Refreshed M season 2026. Local CSV rows now: 250,568
[M] Final: 245,384 rows, 1,363 teams.
Loaded local W CSV: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\source\W_combined_team_box_scores.csv (117,956 rows)
Refreshed W season 2026. Local CSV rows now: 117,956
[W] Final: 114,788 rows, 969 teams.


In [5]:
# -----------------------------------------------------------------
# Canonical men's pipeline setup: name matching, df, massey, paths
# All downstream canonical cells (10-15) draw from variables set here.
# -----------------------------------------------------------------
import re
import numpy as np
import pandas as pd

try:
    from rapidfuzz import process as _rfuzz_process, fuzz as _rfuzz_fuzz
    RAPIDFUZZ_AVAILABLE = True
except Exception:
    RAPIDFUZZ_AVAILABLE = False

ABBR_NAME_FUZZY_MIN = 92.0


def normalize_team_name(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    text = text.replace("&", " and ")
    text = text.replace("'", "")
    text = re.sub(r"\bsaint\b", "st", text)
    text = re.sub(r"\bstate\b", "st", text)
    text = re.sub(r"[^a-z0-9 ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_pipe_values(value: object) -> list:
    if pd.isna(value):
        return []
    return [p.strip() for p in str(value).split("|") if p and str(p).strip()]


# ---- Men's reference data ----
teams_path = source_path / "MTeams.csv"
spellings_path = source_path / "MTeamSpellings.csv"

teams_df = pd.read_csv(teams_path)
spellings_df = (
    pd.read_csv(spellings_path)
    if spellings_path.exists()
    else pd.DataFrame(columns=["TeamID", "TeamNameSpelling"])
)

valid_team_ids = set(pd.to_numeric(teams_df["TeamID"], errors="coerce").dropna().astype(int).tolist())

# Normalized MTeams set (used by ESPN-unmatched diagnostic cell)
_mteams_names = pd.concat(
    [
        teams_df["TeamName"].dropna().astype(str),
        spellings_df["TeamNameSpelling"].dropna().astype(str)
        if "TeamNameSpelling" in spellings_df.columns
        else pd.Series(dtype=str),
    ],
    ignore_index=True,
).map(normalize_team_name)
mteams_norm_set = set(_mteams_names[_mteams_names != ""])

# ---- Alias lookup: MTeams names + spellings ----
alias_to_ids: dict = {}
_tw = teams_df.copy()
_tw["TeamID"] = pd.to_numeric(_tw["TeamID"], errors="coerce")
_tw = _tw.dropna(subset=["TeamID", "TeamName"]).copy()
_tw["TeamID"] = _tw["TeamID"].astype(int)
for _r in _tw.itertuples(index=False):
    _a = normalize_team_name(_r.TeamName)
    if _a:
        alias_to_ids.setdefault(_a, set()).add(int(_r.TeamID))

if not spellings_df.empty and "TeamNameSpelling" in spellings_df.columns:
    _sp = spellings_df.copy()
    _sp["TeamID"] = pd.to_numeric(_sp["TeamID"], errors="coerce")
    _sp = _sp.dropna(subset=["TeamID", "TeamNameSpelling"]).copy()
    _sp["TeamID"] = _sp["TeamID"].astype(int)
    for _r in _sp.itertuples(index=False):
        _a = normalize_team_name(_r.TeamNameSpelling)
        if _a:
            alias_to_ids.setdefault(_a, set()).add(int(_r.TeamID))

# ---- Manual map lookup ----
_manual_map_path = name_matching_path / "m_team_name_master_map.csv"
if not _manual_map_path.exists():
    raise FileNotFoundError(
        f"Missing required manual map: {_manual_map_path}. "
        "Create m_team_name_master_map.csv in name_matching before running."
    )
manual_map_df = pd.read_csv(_manual_map_path).drop_duplicates().reset_index(drop=True)

manual_lookup: dict = {}
_alias_col = next(
    (c for c in ["ESPN_Name", "team_name_norm", "TeamName"] if c in manual_map_df.columns), None
)
if _alias_col is not None and "TeamID" in manual_map_df.columns:
    _mw = manual_map_df.copy()
    _mw["TeamID"] = pd.to_numeric(_mw["TeamID"], errors="coerce")
    _mw = _mw.dropna(subset=[_alias_col, "TeamID"]).copy()
    _mw["TeamID"] = _mw["TeamID"].astype(int)
    _mw = _mw[_mw["TeamID"].isin(valid_team_ids)].copy()
    _mw["alias_norm_key"] = _mw[_alias_col].map(normalize_team_name)
    _mw = _mw[_mw["alias_norm_key"] != ""]
    for _r in _mw.itertuples(index=False):
        manual_lookup.setdefault(_r.alias_norm_key, set()).add(int(_r.TeamID))

# ---- Match men's ESPN team_ids → MTeams KaggleTeamIDs ----
_alias_choices = list(alias_to_ids.keys())
_norm_m = normalized_boxscores["M"].copy()

_name_fields = [
    c for c in [
        "team_short_display_name", "team_display_name",
        "team_location", "team_name", "team_abbreviation",
    ]
    if c in _norm_m.columns
]
team_name_col = _name_fields[0] if _name_fields else None

_norm_m["team_name_norm"] = (
    _norm_m[team_name_col].map(normalize_team_name) if team_name_col else ""
)

_team_id_rows = (
    _norm_m[[c for c in _name_fields + ["team_id", "team_name_norm"] if c in _norm_m.columns]]
    .drop_duplicates(subset=["team_id"])
    .reset_index(drop=True)
)

_match_rows = []
for _r in _team_id_rows.itertuples(index=False):
    _espn_id = _r.team_id
    _cand_norm = []
    for _field in _name_fields:
        for _v in split_pipe_values(getattr(_r, _field, "")):
            _n = normalize_team_name(_v)
            if _n and _n not in _cand_norm:
                _cand_norm.append(_n)

    _kaggle_id = pd.NA
    _method = "unmatched"

    _manual_hits = sorted({t for a in _cand_norm for t in manual_lookup.get(a, set())})
    _exact_hits = sorted({t for a in _cand_norm for t in alias_to_ids.get(a, set())})

    if len(_manual_hits) == 1:
        _kaggle_id = _manual_hits[0]
        _method = "manual_map"
    elif len(_exact_hits) == 1:
        _kaggle_id = _exact_hits[0]
        _method = "exact_alias"
    elif RAPIDFUZZ_AVAILABLE and _alias_choices and _cand_norm:
        _fhits = []
        for _nc in _cand_norm:
            _best = _rfuzz_process.extractOne(_nc, _alias_choices, scorer=_rfuzz_fuzz.WRatio)
            if _best and _best[1] >= ABBR_NAME_FUZZY_MIN:
                _fhits.extend(list(alias_to_ids.get(_best[0], set())))
        _fhits = sorted(set(_fhits))
        if len(_fhits) == 1:
            _kaggle_id = _fhits[0]
            _method = "fuzzy_alias"

    _kaggle_name = pd.NA
    if pd.notna(_kaggle_id):
        _nm = teams_df.loc[teams_df["TeamID"] == int(_kaggle_id), "TeamName"]
        if not _nm.empty:
            _kaggle_name = _nm.iloc[0]

    _match_rows.append({
        "team_id": _espn_id,
        "MTeams_TeamID": _kaggle_id,
        "TeamName": _kaggle_name,
        "match_method": _method,
    })

_match_df_m = pd.DataFrame(_match_rows)

# Persist match to name_matching for Cell 14 (club-share ESPN metadata lookup)
_match_df_m.to_csv(name_matching_path / "m_team_summary_to_teams_matches.csv", index=False)
_match_df_m.to_csv(name_matching_path / "team_summary_to_mteams_matches.csv", index=False)

_id_map = dict(zip(_match_df_m["team_id"], _match_df_m["MTeams_TeamID"]))
_name_map = dict(zip(_match_df_m["team_id"], _match_df_m["TeamName"]))

# Build df: men's normalized boxscores with Kaggle IDs applied
df = _norm_m.copy()
df["KaggleTeamID"] = pd.to_numeric(df["team_id"].map(_id_map), errors="coerce")
df["KaggleOpponentTeamID"] = pd.to_numeric(df["opponent_team_id"].map(_id_map), errors="coerce")
df["KaggleTeamName"] = df["team_id"].map(_name_map)

_matched_share = df["KaggleTeamID"].notna().mean()
print(
    f"[M] Matched {_match_df_m['MTeams_TeamID'].notna().sum():,}/{len(_match_df_m):,} "
    f"unique ESPN teams ({_matched_share:.2%} of rows covered)."
)

# ---- Massey ratings for men's ----
_massey_file = massey_path / TOURNAMENT_CONFIG["M"]["massey_file"]
if not _massey_file.exists():
    raise FileNotFoundError(
        f"Missing Massey file: {_massey_file}.\n"
        "Run massey_pipeline.py with --tournament-type M before running canonical cells."
    )
massey_df = pd.read_csv(_massey_file)
if massey_df.empty:
    raise RuntimeError(f"Massey file is empty: {_massey_file}")
if "Final_Massey_Rating" in massey_df.columns and "Final_Massey" not in massey_df.columns:
    massey_df["Final_Massey"] = pd.to_numeric(massey_df["Final_Massey_Rating"], errors="coerce")

# ---- Key path constants for canonical branch ----
seeds_path = source_path / "MNCAATourneySeeds.csv"
tourney_results_path = source_path / "MNCAATourneyDetailedResults.csv"

print(f"Massey rows: {len(massey_df):,} | seasons: {int(massey_df['Season'].min())}-{int(massey_df['Season'].max())}")
print(f"seeds_path            : {seeds_path}")
print(f"tourney_results_path  : {tourney_results_path}")
print("Canonical men's setup complete.")


[M] Matched 444/1,363 unique ESPN teams (96.71% of rows covered).
Massey rows: 7,808 | seasons: 2005-2026
seeds_path            : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\source\MNCAATourneySeeds.csv
tourney_results_path  : x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\source\MNCAATourneyDetailedResults.csv
Canonical men's setup complete.


In [6]:
# -----------------------------------------------------------------------------
# Persistent unmatched ESPN teams: 2005+ with >=20 games in >=2 seasons
# and no exact MTeams/MTeamSpellings normalized-name match
# -----------------------------------------------------------------------------

season_min = 2005
min_games_per_season = 20
min_seasons = 2

if "team_name_col" not in globals():
    raise RuntimeError("`team_name_col` is not available. Run the mapping cells first.")

if "mteams_norm_set" not in globals():
    mteams_norm = (
        pd.concat(
            [
                teams_df["TeamName"].dropna().astype(str),
                spellings_df["TeamNameSpelling"].dropna().astype(str),
            ],
            ignore_index=True,
        )
        .map(normalize_team_name)
    )
    mteams_norm_set = set(mteams_norm[mteams_norm != ""])

# Team-side unmatched rows only
unmatched_team_rows = df[
    (df["season"] >= season_min)
    & (df["KaggleTeamID"].isna())
    & df["team_name_norm"].notna()
    & (df["team_name_norm"] != "")
] .copy()

unmatched_team_rows["has_exact_mteams_match"] = unmatched_team_rows["team_name_norm"].isin(mteams_norm_set)

# Aggregate games by season + normalized ESPN name
per_season = (
    unmatched_team_rows
    .groupby(["season", "team_name_norm"], as_index=False)
    .agg(
        games=("game_id", "count"),
        sample_name=(team_name_col, lambda s: s.dropna().astype(str).mode().iloc[0] if not s.dropna().empty else ""),
        has_exact_mteams_match=("has_exact_mteams_match", "max"),
    )
)

qualified_seasons = per_season[
    (per_season["games"] >= min_games_per_season)
    & (~per_season["has_exact_mteams_match"])
] .copy()

persistent_unmatched = (
    qualified_seasons
    .groupby("team_name_norm", as_index=False)
    .agg(
        sample_name=("sample_name", lambda s: s.dropna().astype(str).mode().iloc[0] if not s.dropna().empty else ""),
        seasons_count=("season", "nunique"),
        total_games=("games", "sum"),
        seasons=("season", lambda s: ", ".join(str(x) for x in sorted(s.unique()))),
        min_games_in_qualified_season=("games", "min"),
        max_games_in_qualified_season=("games", "max"),
    )
)

persistent_unmatched = persistent_unmatched[persistent_unmatched["seasons_count"] >= min_seasons].copy()
persistent_unmatched = persistent_unmatched.sort_values(["seasons_count", "total_games", "sample_name"], ascending=[False, False, True])

report_out = name_matching_path / "espn_unmatched_persistent_2005plus_20games.csv"
persistent_unmatched.to_csv(report_out, index=False)

print(f"Rows analyzed (unmatched team-side, {season_min}+): {len(unmatched_team_rows):,}")
print(f"Season-name pairs with >= {min_games_per_season} games and no exact MTeams match: {len(qualified_seasons):,}")
print(f"Persistent names (>= {min_seasons} seasons): {len(persistent_unmatched):,}")
print(f"Saved report: {report_out}")

display(persistent_unmatched.head(100))

Rows analyzed (unmatched team-side, 2005+): 8,083
Season-name pairs with >= 20 games and no exact MTeams match: 1
Persistent names (>= 2 seasons): 0
Saved report: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\name_matching\espn_unmatched_persistent_2005plus_20games.csv


,team_name_norm,sample_name,seasons_count,total_games,seasons,min_games_in_qualified_season,max_games_in_qualified_season


In [7]:
# -------------------------------------------
# Expand advanced metrics from team boxscores
# -------------------------------------------

# Keep only rows mapped to Kaggle IDs for feature generation
metrics_df = df[df["KaggleTeamID"].notna()].copy()
metrics_df["KaggleTeamID"] = metrics_df["KaggleTeamID"].astype(int)
metrics_df["KaggleOpponentTeamID"] = pd.to_numeric(metrics_df["KaggleOpponentTeamID"], errors="coerce")


def pick_first_existing(columns: list[str], candidates: list[str]) -> str | None:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


# Resolve schema variations across hoopR seasons/files
team_pf_col = pick_first_existing(metrics_df.columns.tolist(), ["personal_fouls", "fouls", "team_fouls"])
opp_pf_col_src = team_pf_col

if team_pf_col is None:
    print("Warning: no team foul column found; pf_per_poss will be NaN.")

# Build opponent lookups from the second row of each game boxscore pair
opp_cols_needed = [
    "game_id",
    "team_id",
    "team_score",
    "field_goals_made",
    "field_goals_attempted",
    "three_point_field_goals_made",
    "three_point_field_goals_attempted",
    "free_throws_made",
    "free_throws_attempted",
    "offensive_rebounds",
    "defensive_rebounds",
    "total_rebounds",
    "assists",
    "steals",
    "blocks",
    "turnovers",
]
if opp_pf_col_src is not None:
    opp_cols_needed.append(opp_pf_col_src)

opp_cols_available = [col for col in opp_cols_needed if col in metrics_df.columns]

rename_map = {
    "team_id": "opponent_team_id",
    "team_score": "opp_points",
    "field_goals_made": "opp_fgm",
    "field_goals_attempted": "opp_fga",
    "three_point_field_goals_made": "opp_fg3m",
    "three_point_field_goals_attempted": "opp_fg3a",
    "free_throws_made": "opp_ftm",
    "free_throws_attempted": "opp_fta",
    "offensive_rebounds": "opp_orb",
    "defensive_rebounds": "opp_drb",
    "total_rebounds": "opp_trb",
    "assists": "opp_ast",
    "steals": "opp_stl",
    "blocks": "opp_blk",
    "turnovers": "opp_tov",
}
if opp_pf_col_src is not None:
    rename_map[opp_pf_col_src] = "opp_pf"

opponent_stats = metrics_df[opp_cols_available].copy().rename(columns=rename_map)

merge_keys = ["game_id", "opponent_team_id"]
metrics_df = metrics_df.merge(opponent_stats, on=merge_keys, how="left")

# Bring opponent season-end Massey into each game row for SOS features.
# This supports both legacy and current output column names.
massey_rating_candidates = ["Final_Massey_Rating", "Final_Massey"]
massey_rating_col = next((c for c in massey_rating_candidates if c in massey_df.columns), None)

if massey_rating_col is not None:
    massey_lookup = (
        massey_df[["Season", "TeamID", massey_rating_col]]
        .dropna(subset=["Season", "TeamID"])
        .rename(columns={
            "Season": "season",
            "TeamID": "KaggleOpponentTeamID",
            massey_rating_col: "opp_massey_rating",
        })
        .drop_duplicates(["season", "KaggleOpponentTeamID"])
    )

    massey_lookup["season"] = pd.to_numeric(massey_lookup["season"], errors="coerce")
    massey_lookup["KaggleOpponentTeamID"] = pd.to_numeric(massey_lookup["KaggleOpponentTeamID"], errors="coerce")

    metrics_df = metrics_df.merge(
        massey_lookup,
        on=["season", "KaggleOpponentTeamID"],
        how="left",
    )
else:
    metrics_df["opp_massey_rating"] = np.nan
    print("Warning: no Final Massey rating column found in massey_df; opp_massey_rating set to NaN.")

# Create canonical foul fields for downstream formulas
if team_pf_col is not None:
    metrics_df["team_pf"] = metrics_df[team_pf_col]
else:
    metrics_df["team_pf"] = np.nan

if "opp_pf" not in metrics_df.columns:
    metrics_df["opp_pf"] = np.nan

# Convert numeric inputs for safe arithmetic
numeric_cols = [
    "team_score", "field_goals_made", "field_goals_attempted", "three_point_field_goals_made",
    "three_point_field_goals_attempted", "free_throws_made", "free_throws_attempted",
    "offensive_rebounds", "defensive_rebounds", "total_rebounds", "assists", "steals",
    "blocks", "turnovers", "team_pf", "opp_points", "opp_fgm", "opp_fga", "opp_fg3m",
    "opp_fg3a", "opp_ftm", "opp_fta", "opp_orb", "opp_drb", "opp_trb", "opp_ast", "opp_stl",
    "opp_blk", "opp_tov", "opp_pf", "opp_massey_rating",
]
for col in numeric_cols:
    if col in metrics_df.columns:
        metrics_df[col] = pd.to_numeric(metrics_df[col], errors="coerce")


def safe_div(numer, denom):
    return np.where((denom == 0) | pd.isna(denom), np.nan, numer / denom)

# Possessions and ratings (game level)
metrics_df["team_possessions"] = (
    metrics_df["field_goals_attempted"]
    - metrics_df["offensive_rebounds"]
    + metrics_df["turnovers"]
    + 0.475 * metrics_df["free_throws_attempted"]
)
metrics_df["opp_possessions"] = (
    metrics_df["opp_fga"]
    - metrics_df["opp_orb"]
    + metrics_df["opp_tov"]
    + 0.475 * metrics_df["opp_fta"]
)
metrics_df["possessions"] = (metrics_df["team_possessions"] + metrics_df["opp_possessions"]) / 2.0

metrics_df["off_rating"] = 100 * safe_div(metrics_df["team_score"], metrics_df["possessions"])
metrics_df["def_rating"] = 100 * safe_div(metrics_df["opp_points"], metrics_df["possessions"])
metrics_df["net_rating_game"] = metrics_df["off_rating"] - metrics_df["def_rating"]

# Shooting profile metrics
metrics_df["efg_pct"] = safe_div(
    metrics_df["field_goals_made"] + 0.5 * metrics_df["three_point_field_goals_made"],
    metrics_df["field_goals_attempted"],
)
metrics_df["ts_pct"] = safe_div(
    metrics_df["team_score"],
    2 * (metrics_df["field_goals_attempted"] + 0.475 * metrics_df["free_throws_attempted"]),
)

# --- Robust 3-Point Score (Expected Points from 3PT per FGA) ---
# 3PAr (Attempt Rate) = 3PA / FGA
metrics_df["three_par"] = safe_div(metrics_df["three_point_field_goals_attempted"], metrics_df["field_goals_attempted"])
# 3P% = 3PM / 3PA
metrics_df["three_pct"] = safe_div(metrics_df["three_point_field_goals_made"], metrics_df["three_point_field_goals_attempted"])
# Expected points generated by 3-pointers per total Field Goal Attempt
metrics_df["three_score"] = 3 * (metrics_df["three_pct"] * metrics_df["three_par"]) 

# --- Robust Free Throw Score (FT Factor) ---
# FTr (Attempt Rate) = FTA / FGA
metrics_df["ft_par"] = safe_div(metrics_df["free_throws_attempted"], metrics_df["field_goals_attempted"])
# FT% = FTM / FTA
metrics_df["ft_pct"] = safe_div(metrics_df["free_throws_made"], metrics_df["free_throws_attempted"])
# Robust FT Score: FTM / FGA (Captures both getting to the line AND making them)
metrics_df["ft_score"] = safe_div(metrics_df["free_throws_made"], metrics_df["field_goals_attempted"])

# Opponent-side factor rates used for net Four Factors composite
metrics_df["opp_efg_pct"] = safe_div(metrics_df["opp_fgm"] + 0.5 * metrics_df["opp_fg3m"], metrics_df["opp_fga"])
metrics_df["opp_tov_pct"] = safe_div(metrics_df["opp_tov"], metrics_df["opp_possessions"])

# Possession-component metrics
metrics_df["tov_pct"] = safe_div(metrics_df["turnovers"], metrics_df["possessions"])
metrics_df["orb_pct"] = safe_div(metrics_df["offensive_rebounds"], metrics_df["offensive_rebounds"] + metrics_df["opp_drb"])
metrics_df["drb_pct"] = safe_div(metrics_df["defensive_rebounds"], metrics_df["defensive_rebounds"] + metrics_df["opp_orb"])
metrics_df["trb_pct"] = safe_div(metrics_df["total_rebounds"], metrics_df["total_rebounds"] + metrics_df["opp_trb"])

# Playmaking and defensive activity
metrics_df["ast_rate"] = safe_div(metrics_df["assists"], metrics_df["field_goals_made"])
metrics_df["ast_tov_ratio"] = safe_div(metrics_df["assists"], metrics_df["turnovers"])
metrics_df["stl_pct"] = safe_div(metrics_df["steals"], metrics_df["opp_possessions"])
metrics_df["blk_pct"] = safe_div(metrics_df["blocks"], metrics_df["opp_fga"] - metrics_df["opp_fg3a"])
metrics_df["opp_ast_rate"] = safe_div(metrics_df["opp_ast"], metrics_df["opp_fgm"])
metrics_df["opp_stl_pct"] = safe_div(metrics_df["opp_stl"], metrics_df["team_possessions"])
metrics_df["opp_blk_pct"] = safe_div(metrics_df["opp_blk"], metrics_df["field_goals_attempted"] - metrics_df["three_point_field_goals_attempted"])
metrics_df["pf_per_poss"] = safe_div(metrics_df["team_pf"], metrics_df["possessions"])

# Home / away / neutral location tagging
location_col = None
for candidate in ["team_home_away", "home_away", "location", "game_location"]:
    if candidate in metrics_df.columns:
        location_col = candidate
        break

if location_col is not None:
    loc = metrics_df[location_col].astype(str).str.lower()
    metrics_df["loc_tag"] = np.select(
        [loc.str.contains("home") | (loc == "h") | (loc == "vs"),
         loc.str.contains("away") | (loc == "a") | (loc == "@")],
        ["home", "away"],
        default="neutral",
    )
else:
    metrics_df["loc_tag"] = "neutral"

print("Advanced game-level metrics created.")
print(metrics_df[["game_id", "season", "KaggleTeamID", "off_rating", "def_rating", "net_rating_game", "efg_pct", "ts_pct","three_score","ft_score"]].head())

Advanced game-level metrics created.
       game_id  season  KaggleTeamID  off_rating  def_rating  net_rating_game  \
0  250710254.0    2005          1307  114.777618  107.125777         7.651841   
1  250710254.0    2005          1428  107.125777  114.777618        -7.651841   
2  250712638.0    2005          1129  109.762533  128.056288       -18.293755   
3  250712638.0    2005          1431  128.056288  109.762533        18.293755   
4  250710183.0    2005          1452   97.059428  111.865104       -14.805676   

    efg_pct    ts_pct  three_score  ft_score  
0  0.520408  0.558140     0.428571  0.183673  
1  0.500000  0.520446     0.480000  0.120000  
2  0.509434  0.592480     0.339623  0.452830  
3  0.530769  0.581098     0.323077  0.338462  
4  0.443396  0.490644     0.509434  0.226415  


In [8]:
# ------------------------------------------------
# Season-level advanced metrics with H/A adjustments
# ------------------------------------------------
season_col = "season"
season_start, season_end = 2005, 2026

if "season_type" in metrics_df.columns:
    regular_mask = metrics_df["season_type"].astype(str).str.lower().str.contains("regular")
    metrics_regular = metrics_df[regular_mask].copy()
    if metrics_regular.empty:
        metrics_regular = metrics_df.copy()
else:
    metrics_regular = metrics_df.copy()

metrics_regular = metrics_regular[metrics_regular[season_col].between(season_start, season_end)].copy()

base_group_cols = [season_col, "KaggleTeamID", "KaggleTeamName"]
agg_metrics = [
    "possessions", "off_rating", "def_rating", "net_rating_game", "efg_pct", "ts_pct",
    "three_par", "three_pct", "three_score", "ft_par", "ft_pct", "ft_score",
    "tov_pct", "orb_pct", "drb_pct", "trb_pct", "ast_rate",
    "ast_tov_ratio", "stl_pct", "blk_pct", "pf_per_poss", "opp_efg_pct", "opp_tov_pct",
    "opp_ast_rate", "opp_stl_pct", "opp_blk_pct",
]

available_agg_metrics = [col for col in agg_metrics if col in metrics_regular.columns]

extra_aggs = {}
if "opp_massey_rating" in metrics_regular.columns:
    extra_aggs["avg_opp_massey"] = ("opp_massey_rating", "mean")

team_season_advanced = (
    metrics_regular
    .groupby(base_group_cols)
    .agg(
        games_played=("game_id", "count"),
        **{f"avg_{metric}": (metric, "mean") for metric in available_agg_metrics},
        **extra_aggs,
    )
    .reset_index()
)

if "avg_opp_massey" not in team_season_advanced.columns:
    team_season_advanced["avg_opp_massey"] = np.nan

# Home/away/neutral split net ratings
split = (
    metrics_regular
    .groupby(base_group_cols + ["loc_tag"])["net_rating_game"]
    .mean()
    .reset_index()
    .pivot(index=base_group_cols, columns="loc_tag", values="net_rating_game")
    .reset_index()
)
split.columns.name = None
split = split.rename(columns={
    "home": "home_net_rating",
    "away": "away_net_rating",
    "neutral": "neutral_net_rating",
})

team_season_advanced = team_season_advanced.merge(split, on=base_group_cols, how="left")
team_season_advanced["home_away_net_bias"] = team_season_advanced["home_net_rating"] - team_season_advanced["away_net_rating"]
team_season_advanced["adj_net_rating"] = team_season_advanced["avg_net_rating_game"] - 0.5 * team_season_advanced["home_away_net_bias"].fillna(0)

# -----------------------------------------------------------------
# Ceiling Score: average Final Massey of team's top-5 best wins
# -----------------------------------------------------------------
if "Final_Massey_Rating" in massey_df.columns:
    opp_massey_map = (
        massey_df[["Season", "TeamID", "Final_Massey_Rating"]]
        .rename(columns={
            "TeamID": "opponent_team_id",
            "Final_Massey_Rating": "opp_final_massey",
        })
        .drop_duplicates(["Season", "opponent_team_id"])
    )

    metrics_regular = metrics_regular.merge(
        opp_massey_map[["Season", "opponent_team_id", "opp_final_massey"]],
        left_on=["season", "opponent_team_id"],
        right_on=["Season", "opponent_team_id"],
        how="left",
    ).drop(columns=["Season"])
else:
    if "opp_massey_rating" in metrics_regular.columns:
        metrics_regular["opp_final_massey"] = metrics_regular["opp_massey_rating"]
    else:
        metrics_regular["opp_final_massey"] = np.nan

metrics_regular["is_win"] = (metrics_regular["team_score"] > metrics_regular["opp_points"]).astype(int)
wins_only = metrics_regular[metrics_regular["is_win"] == 1].copy()

top_wins = (
    wins_only.sort_values(["season", "KaggleTeamID", "opp_final_massey"], ascending=[True, True, False])
    .groupby(["season", "KaggleTeamID"])
    .head(5)
)

ceiling_score_df = (
    top_wins.groupby(["season", "KaggleTeamID"])
    .agg(ceiling_score=("opp_final_massey", "mean"))
    .reset_index()
)

team_season_advanced = team_season_advanced.merge(
    ceiling_score_df,
    left_on=["season", "KaggleTeamID"],
    right_on=["season", "KaggleTeamID"],
    how="left",
)

if "Final_Massey_Rating" in massey_df.columns:
    min_massey = pd.to_numeric(massey_df["Final_Massey_Rating"], errors="coerce").min()
elif "Final_Massey" in massey_df.columns:
    min_massey = pd.to_numeric(massey_df["Final_Massey"], errors="coerce").min()
else:
    min_massey = np.nan

team_season_advanced["ceiling_score"] = team_season_advanced["ceiling_score"].fillna(min_massey)

In [9]:
# -----------------------------------------------------------------------
# Synthetic metrics: two-pass calibrated composite construction
# Pass 1: provisional equal-weight composites for logistic calibration.
# Pass 2: recompute composites with learned weights; all exports use Pass 2.
# -----------------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ---- Shared helpers ----

def zscore_series(series):
    sigma = series.std(ddof=0)
    if pd.isna(sigma) or sigma == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / sigma


def compute_composite_weights(mdf: pd.DataFrame, factor_diff_cols: list) -> dict:
    """Fit logistic regression on standardised factor diffs vs Team1Win.
    Returns: weights (normalised signed), raw_coef, auc.
    Falls back to equal weights when data are insufficient.
    """
    valid = [c for c in factor_diff_cols if c in mdf.columns]
    n = len(factor_diff_cols)
    fallback = {"weights": np.ones(n) / n, "raw_coef": np.ones(n) / n, "auc": np.nan}
    if len(valid) < 2:
        return fallback
    subset = mdf.dropna(subset=valid + ["Team1Win"]).copy()
    if len(subset) < 50:
        return fallback
    X = subset[valid].values
    y = subset["Team1Win"].values
    sc = StandardScaler()
    Xs = sc.fit_transform(X)
    clf = LogisticRegression(fit_intercept=False, random_state=42, max_iter=500)
    clf.fit(Xs, y)
    raw_coef = clf.coef_[0]
    total_abs = np.sum(np.abs(raw_coef))
    weights = raw_coef / total_abs if total_abs > 0 else fallback["weights"]
    auc = roc_auc_score(y, clf.predict_proba(Xs)[:, 1])
    return {"weights": weights, "raw_coef": raw_coef, "auc": auc}


def _build_prov_model_df(ts_adv: pd.DataFrame, t_res_path, s_start: int, s_end: int, s_col: str) -> pd.DataFrame:
    """Build a minimal provisional model_df from team_season_advanced for weight calibration.
    Uses lower TeamID as Team1 (deterministic, avoids needing seeds at this stage).
    """
    comp_cols = [c for c in [
        "net_efg", "net_tov", "net_reb",
        "net_ast_rate", "net_stl_pct", "net_blk_pct",
    ] if c in ts_adv.columns]
    ts_slim = (
        ts_adv[[s_col, "KaggleTeamID"] + comp_cols]
        .rename(columns={s_col: "Season", "KaggleTeamID": "TeamID"})
        .dropna(subset=["Season", "TeamID"])
        .copy()
    )
    tourney = pd.read_csv(t_res_path, usecols=["Season", "WTeamID", "LTeamID"])
    tourney = tourney[tourney["Season"].between(s_start, s_end)].copy()
    tourney["Team1ID"] = np.where(
        tourney["WTeamID"] <= tourney["LTeamID"], tourney["WTeamID"], tourney["LTeamID"]
    )
    tourney["Team2ID"] = np.where(
        tourney["WTeamID"] <= tourney["LTeamID"], tourney["LTeamID"], tourney["WTeamID"]
    )
    tourney["Team1Win"] = (tourney["WTeamID"] == tourney["Team1ID"]).astype(int)
    t1 = ts_slim.rename(columns={"TeamID": "Team1ID", **{c: f"{c}_t1" for c in comp_cols}})
    t2 = ts_slim.rename(columns={"TeamID": "Team2ID", **{c: f"{c}_t2" for c in comp_cols}})
    prov = (
        tourney
        .merge(t1, on=["Season", "Team1ID"], how="left")
        .merge(t2, on=["Season", "Team2ID"], how="left")
    )
    for c in comp_cols:
        prov[f"Diff_{c}"] = prov[f"{c}_t1"] - prov[f"{c}_t2"]
    return prov


# Factor column names as they appear in model_df Diff_ form
THREE_FACTOR_COLS = ["Diff_net_efg", "Diff_net_tov", "Diff_net_reb"]
PLAYMAKING_COLS   = ["Diff_net_ast_rate", "Diff_net_stl_pct", "Diff_net_blk_pct"]


# ================================================================
# 1) Pythagorean win expectancy from season-average efficiencies
# ================================================================
pyth_exp = 11.5
off_pow = np.where(
    team_season_advanced["avg_off_rating"] > 0,
    np.power(team_season_advanced["avg_off_rating"], pyth_exp),
    np.nan,
)
def_pow = np.where(
    team_season_advanced["avg_def_rating"] > 0,
    np.power(team_season_advanced["avg_def_rating"], pyth_exp),
    np.nan,
)
team_season_advanced["pyth_win_pct"] = safe_div(off_pow, off_pow + def_pow)

# 2) SOS-adjusted net rating using average opponent final Massey
team_season_advanced["sos_adj_net_rating"] = (
    team_season_advanced["adj_net_rating"] + team_season_advanced["avg_opp_massey"]
)

print(f"SOS-adjusted net rating: {team_season_advanced['sos_adj_net_rating'].mean(skipna=True):.2f}")
print(f"Adj net rating:          {team_season_advanced['adj_net_rating'].mean(skipna=True):.2f}")

# ================================================================
# 3) Four-Factors component metrics
# ================================================================
for _src, _dst in [
    ("avg_opp_efg_pct", "opp_avg_efg_pct"),
    ("avg_opp_tov_pct", "opp_avg_tov_pct"),
    ("avg_opp_ast_rate", "opp_avg_ast_rate"),
    ("avg_opp_stl_pct", "opp_avg_stl_pct"),
    ("avg_opp_blk_pct", "opp_avg_blk_pct"),
]:
    team_season_advanced[_dst] = (
        team_season_advanced[_src].copy() if _src in team_season_advanced.columns else np.nan
    )

team_season_advanced["net_efg"] = team_season_advanced["avg_efg_pct"] - team_season_advanced["opp_avg_efg_pct"]
team_season_advanced["net_tov"] = team_season_advanced["opp_avg_tov_pct"] - team_season_advanced["avg_tov_pct"]
team_season_advanced["net_reb"] = team_season_advanced["avg_orb_pct"] + team_season_advanced["avg_drb_pct"]

team_season_advanced["net_efg_z"] = zscore_series(team_season_advanced["net_efg"])
team_season_advanced["net_tov_z"] = zscore_series(team_season_advanced["net_tov"])
team_season_advanced["net_reb_z"] = zscore_series(team_season_advanced["net_reb"])

# ================================================================
# 4) Playmaking + defense components
# ================================================================
team_season_advanced["net_ast_rate"] = team_season_advanced["avg_ast_rate"] - team_season_advanced["opp_avg_ast_rate"]
team_season_advanced["net_stl_pct"]  = team_season_advanced["avg_stl_pct"]  - team_season_advanced["opp_avg_stl_pct"]
team_season_advanced["net_blk_pct"]  = team_season_advanced["avg_blk_pct"]  - team_season_advanced["opp_avg_blk_pct"]

team_season_advanced["net_ast_rate_z"] = zscore_series(team_season_advanced["net_ast_rate"])
team_season_advanced["net_stl_pct_z"]  = zscore_series(team_season_advanced["net_stl_pct"])
team_season_advanced["net_blk_pct_z"]  = zscore_series(team_season_advanced["net_blk_pct"])

# ================================================================
# PASS 1 — provisional equal-weight composites (for calibration)
# ================================================================
team_season_advanced["three_factors_composite"] = (
    team_season_advanced["net_efg_z"]
    + team_season_advanced["net_tov_z"]
    + team_season_advanced["net_reb_z"]
) / 3.0

team_season_advanced["playmaking_defense_composite"] = (
    team_season_advanced["net_ast_rate_z"]
    + team_season_advanced["net_stl_pct_z"]
    + team_season_advanced["net_blk_pct_z"]
) / 3.0

# ================================================================
# WEIGHT CALIBRATION — build provisional model_df, fit weights
# ================================================================
_prov_mdf = _build_prov_model_df(
    team_season_advanced, tourney_results_path, season_start, season_end, season_col
)

_result_tf = compute_composite_weights(_prov_mdf, THREE_FACTOR_COLS)
_result_pd = compute_composite_weights(_prov_mdf, PLAYMAKING_COLS)

weights_three_factors = _result_tf["weights"]
weights_playmaking    = _result_pd["weights"]

print(f"\n-- Calibrated Three-Factors Weights (provisional AUC: {_result_tf['auc']:.4f}) --")
for _lbl, _w in zip(["net_efg", "net_tov", "net_reb"], weights_three_factors):
    print(f"  {_lbl}: {_w:+.4f}")

print(f"\n-- Calibrated Playmaking-Defense Weights (provisional AUC: {_result_pd['auc']:.4f}) --")
for _lbl, _w in zip(["net_ast_rate", "net_stl_pct", "net_blk_pct"], weights_playmaking):
    print(f"  {_lbl}: {_w:+.4f}")

# ================================================================
# PASS 2 — apply calibrated weights to final composites
# ================================================================
team_season_advanced["three_factors_composite"] = (
    weights_three_factors[0] * team_season_advanced["net_efg_z"]
    + weights_three_factors[1] * team_season_advanced["net_tov_z"]
    + weights_three_factors[2] * team_season_advanced["net_reb_z"]
)

team_season_advanced["playmaking_defense_composite"] = (
    weights_playmaking[0] * team_season_advanced["net_ast_rate_z"]
    + weights_playmaking[1] * team_season_advanced["net_stl_pct_z"]
    + weights_playmaking[2] * team_season_advanced["net_blk_pct_z"]
)

# Emit weight diagnostics artifact
_weight_records = (
    [
        {"composite": "three_factors", "component": c.replace("Diff_", ""), "weight": float(w)}
        for c, w in zip(THREE_FACTOR_COLS, weights_three_factors)
    ]
    + [
        {"composite": "playmaking_defense", "component": c.replace("Diff_", ""), "weight": float(w)}
        for c, w in zip(PLAYMAKING_COLS, weights_playmaking)
    ]
)
weight_diag_path = modeling_diag_path / "calibrated_composite_weights.csv"
pd.DataFrame(_weight_records).to_csv(weight_diag_path, index=False)
print(f"\nCalibrated weight diagnostics saved: {weight_diag_path}")

# ================================================================
# Bring in seed features
# ================================================================
seeds_df = pd.read_csv(seeds_path)
seeds_df = seeds_df[["Season", "TeamID", "Seed"]].copy()

supplemental_seed_candidates = [
    processed_path / "club_share" / "2026_Seeding.xlsx",
    source_path / "2026_Seeding.xlsx",
]
supplemental_seed_path = next((p for p in supplemental_seed_candidates if p.exists()), None)
if supplemental_seed_path is not None:
    supplemental_seeds = pd.read_excel(supplemental_seed_path, sheet_name="MNCAATourneySeeds")
    supplemental_seeds = supplemental_seeds[["Season", "TeamID", "Seed"]].copy()
    seeds_df = (
        pd.concat([seeds_df, supplemental_seeds], ignore_index=True)
        .dropna(subset=["Season", "TeamID"])
        .copy()
    )
    print(f"Loaded supplemental seeds from: {supplemental_seed_path}")

seeds_df["Season"] = pd.to_numeric(seeds_df["Season"], errors="coerce")
seeds_df["TeamID"] = pd.to_numeric(seeds_df["TeamID"], errors="coerce")
seeds_df = seeds_df.dropna(subset=["Season", "TeamID"]).copy()
seeds_df["Season"] = seeds_df["Season"].astype(int)
seeds_df["TeamID"] = seeds_df["TeamID"].astype(int)
seeds_df["Seed"] = seeds_df["Seed"].astype(str).str.strip()
seeds_df = seeds_df[seeds_df["Seed"] != ""].copy()
seeds_df = seeds_df.drop_duplicates(subset=["Season", "TeamID"], keep="last").copy()

seeds_df["SeedNumBase"] = seeds_df["Seed"].str.extract(r"(\d{2})").astype(float)
seeds_df["SeedPlayInSuffix"] = seeds_df["Seed"].str.extract(r"([a-z])", expand=False).fillna("").str.lower()
seeds_df["SeedNum"] = seeds_df["SeedNumBase"] + np.where(seeds_df["SeedPlayInSuffix"] == "b", 0.5, 0.0)
seeds_df["SeedRegion"] = seeds_df["Seed"].str[0]
seeds_df["SeedPlayInFlag"] = seeds_df["SeedPlayInSuffix"].ne("").astype(int)

# ================================================================
# Merge season-advanced + massey + seeds -> team_features
# ================================================================
team_features = (
    team_season_advanced
    .rename(columns={season_col: "Season", "KaggleTeamID": "TeamID", "KaggleTeamName": "TeamName"})
    .merge(massey_df, on=["Season", "TeamID"], how="left")
    .merge(
        seeds_df[["Season", "TeamID", "Seed", "SeedNum", "SeedNumBase",
                  "SeedPlayInSuffix", "SeedRegion", "SeedPlayInFlag"]],
        on=["Season", "TeamID"],
        how="left",
    )
)

team_features = team_features[team_features["Season"].between(season_start, season_end)].copy()

# Preserve observed seed numeric.
team_features["SeedNum_Observed"] = team_features["SeedNum"]
team_features["SeedWasImputed"] = 0

# ================================================================
# 4) Tournament prestige (5-year rolling decay)
# ================================================================
tourney_wins = (
    pd.read_csv(tourney_results_path, usecols=["Season", "WTeamID"])
    .groupby(["Season", "WTeamID"])
    .size()
    .reset_index(name="TournamentWins")
    .rename(columns={"WTeamID": "TeamID"})
)

for lag in range(1, 6):
    lagged = tourney_wins[["Season", "TeamID", "TournamentWins"]].copy()
    lagged["Season"] = lagged["Season"] + lag
    lagged = lagged.rename(columns={"TournamentWins": f"TournamentWins_lag{lag}"})
    team_features = team_features.merge(lagged, on=["Season", "TeamID"], how="left")

lag_cols = [f"TournamentWins_lag{lag}" for lag in range(1, 6)]
for col in lag_cols:
    team_features[col] = team_features[col].fillna(0)

team_features["TournamentPrestige_5Y_Decay"] = sum(
    team_features[f"TournamentWins_lag{lag}"] / (2 ** (lag - 1))
    for lag in range(1, 6)
)

# ================================================================
# 5) Reliability-weighted momentum
# ================================================================
if "Relative_Massey_Momentum" in team_features.columns:
    team_features["weighted_massey_momentum"] = (
        team_features["Relative_Massey_Momentum"] * np.log1p(team_features["games_played"])
    )
else:
    team_features["weighted_massey_momentum"] = np.nan

# ================================================================
# 6) Seed-adjusted Massey and Massey-adjusted Seed
# ================================================================
massey_key_col = (
    "Final_Massey_Rating" if "Final_Massey_Rating" in team_features.columns
    else ("Final_Massey" if "Final_Massey" in team_features.columns else None)
)
slope_seed = np.nan
intercept_seed = np.nan

if massey_key_col is not None and "SeedNum" in team_features.columns:
    seed_fit = team_features[[massey_key_col, "SeedNum"]].dropna().copy()

    if len(seed_fit) >= 2:
        slope_seed, intercept_seed = np.polyfit(seed_fit[massey_key_col], seed_fit["SeedNum"], 1)
        team_features["Expected_Seed"] = intercept_seed + slope_seed * team_features[massey_key_col]
        team_features["Adj_Seed"] = np.round(np.clip(team_features["Expected_Seed"], 1, 16))
    else:
        team_features["Expected_Seed"] = np.nan
        team_features["Adj_Seed"] = np.nan

    inferred_base = np.floor(team_features["SeedNum"])
    inferred_suffix = np.where(np.isclose(team_features["SeedNum"] - inferred_base, 0.5), "b", "")
    inferred_suffix_series = pd.Series(inferred_suffix, index=team_features.index, dtype="object")

    seed_base_fill_mask = team_features["SeedNumBase"].isna() & team_features["SeedNum"].notna()
    suffix_fill_mask    = team_features["SeedPlayInSuffix"].isna() & team_features["SeedNum"].notna()
    flag_fill_mask      = team_features["SeedPlayInFlag"].isna() & team_features["SeedNum"].notna()

    team_features.loc[seed_base_fill_mask, "SeedNumBase"] = inferred_base.loc[seed_base_fill_mask]
    team_features.loc[suffix_fill_mask, "SeedPlayInSuffix"] = inferred_suffix_series.loc[suffix_fill_mask]
    team_features.loc[flag_fill_mask, "SeedPlayInFlag"] = np.where(
        inferred_suffix_series.loc[flag_fill_mask] == "", 0, 1
    )
    team_features.loc[
        team_features["SeedRegion"].isna() & team_features["SeedNum"].notna(), "SeedRegion"
    ] = "I"

    def build_seed_label(row):
        if pd.notna(row.get("Seed")) and str(row.get("Seed")).strip() != "":
            return str(row.get("Seed")).strip()
        if pd.isna(row.get("SeedNum")):
            return pd.NA
        seed_num = float(row.get("SeedNum"))
        base = int(np.floor(seed_num))
        suffix = "b" if np.isclose(seed_num - base, 0.5) else ""
        region = str(row.get("SeedRegion", "I")).strip() or "I"
        return f"{region[0].upper()}{base:02d}{suffix}"

    missing_seed_label_mask = (
        team_features["Seed"].isna() | (team_features["Seed"].astype(str).str.strip() == "")
    )
    team_features.loc[missing_seed_label_mask, "Seed"] = (
        team_features.loc[missing_seed_label_mask].apply(build_seed_label, axis=1)
    )

    expected_massey_by_seed = seed_fit.groupby("SeedNum")[massey_key_col].mean()
    team_features["Expected_Massey_For_Seed"] = team_features["SeedNum"].map(expected_massey_by_seed)

    if pd.notna(slope_seed) and not np.isclose(slope_seed, 0.0):
        fallback_mask = (
            team_features["Expected_Massey_For_Seed"].isna() & team_features["SeedNum"].notna()
        )
        team_features.loc[fallback_mask, "Expected_Massey_For_Seed"] = (
            team_features.loc[fallback_mask, "SeedNum"] - intercept_seed
        ) / slope_seed

    team_features["Adj_Massey"] = team_features[massey_key_col] - team_features["Expected_Massey_For_Seed"]
else:
    team_features["Expected_Seed"] = np.nan
    team_features["Adj_Seed"] = np.nan
    team_features["Expected_Massey_For_Seed"] = np.nan
    team_features["Adj_Massey"] = np.nan

# Canonicalize TeamName after merges
if "TeamName" not in team_features.columns:
    if "TeamName_x" in team_features.columns and "TeamName_y" in team_features.columns:
        team_features["TeamName"] = team_features["TeamName_x"].combine_first(team_features["TeamName_y"])
    elif "TeamName_x" in team_features.columns:
        team_features["TeamName"] = team_features["TeamName_x"]
    elif "TeamName_y" in team_features.columns:
        team_features["TeamName"] = team_features["TeamName_y"]
team_features = team_features.drop(
    columns=[c for c in ["TeamName_x", "TeamName_y"] if c in team_features.columns]
)

# ================================================================
# Team-feature completeness diagnostics by season
# ================================================================
key_feature_cols = [
    "adj_net_rating", "ceiling_score", "pyth_win_pct", "sos_adj_net_rating",
    "three_factors_composite", "weighted_massey_momentum", "Massey_Momentum",
    "TournamentPrestige_5Y_Decay", "Adj_Seed", "Adj_Massey", "SeedNum",
]
if massey_key_col is not None:
    key_feature_cols.insert(1, massey_key_col)

present_key_cols = [c for c in key_feature_cols if c in team_features.columns]


def compute_non_null_share(frame: pd.DataFrame, cols: list) -> float:
    if not cols:
        return np.nan
    return frame[cols].notna().mean().mean()


completeness_by_season = (
    team_features.groupby("Season")
    .apply(
        lambda g: pd.Series({
            "team_rows": len(g),
            "games_played_median": g["games_played"].median() if "games_played" in g.columns else np.nan,
            "games_played_p10": g["games_played"].quantile(0.10) if "games_played" in g.columns else np.nan,
            "games_played_p90": g["games_played"].quantile(0.90) if "games_played" in g.columns else np.nan,
            "adj_net_non_null_share": g["adj_net_rating"].notna().mean() if "adj_net_rating" in g.columns else np.nan,
            "massey_non_null_share": g[massey_key_col].notna().mean() if massey_key_col is not None else np.nan,
            "seed_non_null_share": g["SeedNum"].notna().mean() if "SeedNum" in g.columns else np.nan,
            "seed_imputed_share": g["SeedWasImputed"].mean() if "SeedWasImputed" in g.columns else np.nan,
            "key_feature_non_null_share": compute_non_null_share(g, present_key_cols),
        })
    )
    .reset_index()
    .sort_values("Season")
)

completeness_out = modeling_diag_path / "team_feature_completeness_by_season.csv"
completeness_by_season.to_csv(completeness_out, index=False)

team_features_out = modeling_path / "team_season_model_features.csv"
team_features.to_csv(team_features_out, index=False)

preview_cols = [
    c for c in [
        "Season", "TeamID", "TeamName", "Seed", "SeedNum", "SeedNum_Observed", "SeedWasImputed",
        massey_key_col, "Massey_Momentum",
        "TournamentPrestige_5Y_Decay", "Expected_Seed", "Adj_Seed",
        "Expected_Massey_For_Seed", "Adj_Massey",
        "pyth_win_pct", "sos_adj_net_rating", "three_factors_composite",
        "weighted_massey_momentum", "avg_three_score", "avg_ft_par", "avg_ft_score", "adj_net_rating",
    ]
    if c in team_features.columns and c is not None
]

print(f"\nSaved team-season features: {team_features_out}")
print(f"Saved completeness audit:   {completeness_out}")
if "SeedWasImputed" in team_features.columns:
    print(f"Rows with imputed seeds: {int(team_features['SeedWasImputed'].sum()):,}")
print(team_features[preview_cols].head())
print("Completeness preview:")
print(completeness_by_season.tail(10))


SOS-adjusted net rating: -7.16
Adj net rating:          -7.66

-- Calibrated Three-Factors Weights (provisional AUC: 0.7129) --
  net_efg: +0.3596
  net_tov: +0.3368
  net_reb: +0.3036

-- Calibrated Playmaking-Defense Weights (provisional AUC: 0.6402) --
  net_ast_rate: +0.2973
  net_stl_pct: +0.3168
  net_blk_pct: +0.3859

Calibrated weight diagnostics saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\calibrated_composite_weights.csv
Loaded supplemental seeds from: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\2026_Seeding.xlsx

Saved team-season features: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\team_season_model_features.csv
Saved completeness audit:   x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\

In [10]:
# ---------------------------------------------------------------
# 2026 seeding enrichment: retain AlternateTeamID from Excel slots
# ---------------------------------------------------------------
seed_sheet_name = "MNCAATourneySeeds"
seed_file_candidates = [
    processed_path / "club_share" / "2026_Seeding.xlsx",
    source_path / "2026_Seeding.xlsx",
]
seed_file_path = next((p for p in seed_file_candidates if p.exists()), None)

if seed_file_path is not None:
    seed_slots_2026 = pd.read_excel(seed_file_path, sheet_name=seed_sheet_name)

    if "AlternateTeamID" not in seed_slots_2026.columns:
        seed_slots_2026["AlternateTeamID"] = np.nan

    seed_slots_2026 = seed_slots_2026[["Season", "Seed", "TeamID", "AlternateTeamID"]].copy()
    seed_slots_2026["Season"] = pd.to_numeric(seed_slots_2026["Season"], errors="coerce")
    seed_slots_2026["TeamID"] = pd.to_numeric(seed_slots_2026["TeamID"], errors="coerce")
    seed_slots_2026["AlternateTeamID"] = pd.to_numeric(seed_slots_2026["AlternateTeamID"], errors="coerce")
    seed_slots_2026["Seed"] = seed_slots_2026["Seed"].astype(str).str.strip()
    seed_slots_2026 = seed_slots_2026.dropna(subset=["Season", "Seed", "TeamID"]).copy()
    seed_slots_2026 = seed_slots_2026[seed_slots_2026["Season"] == 2026].copy()
    seed_slots_2026["Season"] = seed_slots_2026["Season"].astype(int)
    seed_slots_2026["TeamID"] = seed_slots_2026["TeamID"].astype(int)
    seed_slots_2026 = seed_slots_2026.drop_duplicates(subset=["Season", "Seed"], keep="last").copy()

    seed_alt_lookup = seed_slots_2026[["Season", "Seed", "AlternateTeamID"]].copy()

    seeds_df = seeds_df.merge(seed_alt_lookup, on=["Season", "Seed"], how="left", suffixes=("", "_slot"))
    if "AlternateTeamID_slot" in seeds_df.columns:
        seeds_df["AlternateTeamID"] = seeds_df["AlternateTeamID"].combine_first(seeds_df["AlternateTeamID_slot"])
        seeds_df = seeds_df.drop(columns=["AlternateTeamID_slot"])

    team_features = team_features.merge(seed_alt_lookup, on=["Season", "Seed"], how="left", suffixes=("", "_slot"))
    if "AlternateTeamID_slot" in team_features.columns:
        team_features["AlternateTeamID"] = team_features["AlternateTeamID"].combine_first(team_features["AlternateTeamID_slot"])
        team_features = team_features.drop(columns=["AlternateTeamID_slot"])

    seeds_df["AlternateTeamID"] = pd.to_numeric(seeds_df.get("AlternateTeamID", np.nan), errors="coerce")
    team_features["AlternateTeamID"] = pd.to_numeric(team_features.get("AlternateTeamID", np.nan), errors="coerce")

    if "team_features_out" in globals():
        team_features.to_csv(team_features_out, index=False)

    print(f"Loaded 2026 alternate seed slots from: {seed_file_path}")
    print(f"Seed slots with AlternateTeamID: {int(seed_slots_2026['AlternateTeamID'].notna().sum())}")
else:
    print("No 2026 seeding Excel found for AlternateTeamID enrichment.")

Loaded 2026 alternate seed slots from: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\2026_Seeding.xlsx
Seed slots with AlternateTeamID: 4


In [11]:
# ------------------------------------------------
# Build finalized tournament matchup modeling table
# ------------------------------------------------
season_start, season_end = 2005, 2026
tourney_results = pd.read_csv(tourney_results_path)
tourney_results = tourney_results[["Season", "DayNum", "WTeamID", "LTeamID", "WScore", "LScore", "WLoc"]].copy()
tourney_results = tourney_results[tourney_results["Season"].between(season_start, season_end)].copy()

# Join team-season features for each side
feature_table = team_features[team_features["Season"].between(season_start, season_end)].copy()

# Determine ranking columns used to orient Team1/Team2
seed_col = "SeedNum" if "SeedNum" in feature_table.columns else None
massey_candidates = ["Final_Massey", "Final_Massey_Rating"]
massey_col = next((c for c in massey_candidates if c in feature_table.columns), None)

rank_cols = ["Season", "TeamID"]
if seed_col is not None:
    rank_cols.append(seed_col)
if massey_col is not None:
    rank_cols.append(massey_col)

rank_table = feature_table[rank_cols].drop_duplicates(["Season", "TeamID"])

w_rank = rank_table.rename(columns={
    "TeamID": "WTeamID",
    seed_col: "W_SeedNum" if seed_col is not None else seed_col,
    massey_col: "W_Massey" if massey_col is not None else massey_col,
})
l_rank = rank_table.rename(columns={
    "TeamID": "LTeamID",
    seed_col: "L_SeedNum" if seed_col is not None else seed_col,
    massey_col: "L_Massey" if massey_col is not None else massey_col,
})

tourney_ranked = (
    tourney_results
    .merge(w_rank, on=["Season", "WTeamID"], how="left")
    .merge(l_rank, on=["Season", "LTeamID"], how="left")
)

# Team1 orientation rule:
# 1) Seed first: Team1 is the better seed (lower numeric seed, e.g., 1 over 16)
# 2) If same seed, Team1 has higher Massey rating
# 3) If still tied/missing, fallback to lower TeamID for deterministic ordering
team1_is_w = pd.Series(index=tourney_ranked.index, dtype=object)

if seed_col is not None:
    seed_available = tourney_ranked["W_SeedNum"].notna() & tourney_ranked["L_SeedNum"].notna()
    seed_differs = seed_available & (tourney_ranked["W_SeedNum"] != tourney_ranked["L_SeedNum"])
    team1_is_w.loc[seed_differs] = (
        tourney_ranked.loc[seed_differs, "W_SeedNum"] <= tourney_ranked.loc[seed_differs, "L_SeedNum"]
    )

if massey_col is not None:
    undecided = team1_is_w.isna()
    same_seed_or_missing = undecided
    if seed_col is not None:
        same_seed_or_missing = undecided & (
            (~(tourney_ranked["W_SeedNum"].notna() & tourney_ranked["L_SeedNum"].notna()))
            | (tourney_ranked["W_SeedNum"] == tourney_ranked["L_SeedNum"])
        )

    massey_available = tourney_ranked["W_Massey"].notna() & tourney_ranked["L_Massey"].notna()
    massey_differs = same_seed_or_missing & massey_available & (tourney_ranked["W_Massey"] != tourney_ranked["L_Massey"])
    team1_is_w.loc[massey_differs] = (
        tourney_ranked.loc[massey_differs, "W_Massey"] >= tourney_ranked.loc[massey_differs, "L_Massey"]
    )

fallback = team1_is_w.isna()
team1_is_w.loc[fallback] = (
    tourney_ranked.loc[fallback, "WTeamID"] <= tourney_ranked.loc[fallback, "LTeamID"]
)

team1_is_w = team1_is_w.astype(bool)
tourney_results["Team1ID"] = np.where(team1_is_w, tourney_results["WTeamID"], tourney_results["LTeamID"])
tourney_results["Team2ID"] = np.where(team1_is_w, tourney_results["LTeamID"], tourney_results["WTeamID"] )
tourney_results["Team1Win"] = (tourney_results["WTeamID"] == tourney_results["Team1ID"]).astype("Int64")

team1_features = feature_table.add_prefix("Team1_").rename(columns={
    "Team1_Season": "Season",
    "Team1_TeamID": "Team1ID",
})
team2_features = feature_table.add_prefix("Team2_").rename(columns={
    "Team2_Season": "Season",
    "Team2_TeamID": "Team2ID",
})

model_df = (
    tourney_results
    .merge(team1_features, on=["Season", "Team1ID"], how="left")
    .merge(team2_features, on=["Season", "Team2ID"], how="left")
)

# Difference features for numeric columns (Team1 - Team2)
numeric_cols_t1 = [
    col for col in model_df.columns
    if col.startswith("Team1_") and pd.api.types.is_numeric_dtype(model_df[col])
]

for col_t1 in numeric_cols_t1:
    col_t2 = col_t1.replace("Team1_", "Team2_", 1)
    if col_t2 in model_df.columns:
        diff_name = "Diff_" + col_t1.replace("Team1_", "")
        model_df[diff_name] = model_df[col_t1] - model_df[col_t2]

# Keep Diff_SeedNum positive while Team1 is better seed (e.g., 1 vs 16 => +15)
if "Team1_SeedNum" in model_df.columns and "Team2_SeedNum" in model_df.columns:
    model_df["Diff_SeedNum"] = model_df["Team2_SeedNum"] - model_df["Team1_SeedNum"]


# Games played discrepancy diagnostics (leakage guardrails)
if "Team1_games_played" in model_df.columns and "Team2_games_played" in model_df.columns:
    gp_diag = model_df[[
        "Season", "DayNum", "WTeamID", "LTeamID", "Team1ID", "Team2ID", "Team1Win",
        "Team1_TeamName", "Team2_TeamName", "Team1_games_played", "Team2_games_played",
        "Team1_Seed", "Team2_Seed",
    ]].copy()
    gp_diag["games_played_diff"] = gp_diag["Team1_games_played"] - gp_diag["Team2_games_played"]
    gp_diag["games_played_abs_diff"] = gp_diag["games_played_diff"].abs()

    top_gp_discrepancies = gp_diag.sort_values(
        ["games_played_abs_diff", "Season", "DayNum"], ascending=[False, True, True]
    ).head(100)

    top_gp_out = modeling_diag_path / "top_games_played_discrepancies.csv"
    top_gp_discrepancies.to_csv(top_gp_out, index=False)

    gp_season_span = (
        feature_table.groupby("Season")
        .agg(
            teams=("TeamID", "nunique"),
            min_games_played=("games_played", "min"),
            p10_games_played=("games_played", lambda s: s.quantile(0.10)),
            median_games_played=("games_played", "median"),
            p90_games_played=("games_played", lambda s: s.quantile(0.90)),
            max_games_played=("games_played", "max"),
        )
        .reset_index()
        .sort_values("Season")
    )
    gp_span_out = modeling_diag_path / "games_played_season_span.csv"
    gp_season_span.to_csv(gp_span_out, index=False)

    print(f"Saved top games-played discrepancies: {top_gp_out}")
    print(f"Saved games-played season span audit: {gp_span_out}")
    print("Top games-played discrepancy preview:")
    print(top_gp_discrepancies.head(15).to_string(index=False))

# Keep a clean modeling table
id_cols = [
    "Season", "DayNum", "Team1ID", "Team2ID", "Team1Win",
    "WTeamID", "LTeamID", "WScore", "LScore", "WLoc",
]
meta_cols = [
    "Team1_TeamName", "Team2_TeamName", "Team1_Seed", "Team2_Seed",
    "Team1_SeedRegion", "Team2_SeedRegion",
]
diff_cols = [col for col in model_df.columns if col.startswith("Diff_")]
final_cols = [col for col in id_cols + meta_cols + diff_cols if col in model_df.columns]

modeling_dataset = model_df[final_cols].copy()

# Missing target diagnostics
missing_target_rows = modeling_dataset[modeling_dataset["Team1Win"].isna()].copy()
missing_target_out = modeling_diag_path / "modeling_dataset_missing_target_rows.csv"
missing_target_rows.to_csv(missing_target_out, index=False)

# Row-level data quality flags
if diff_cols:
    modeling_dataset["diff_missing_count"] = modeling_dataset[diff_cols].isna().sum(axis=1)
    modeling_dataset["has_all_diff_features"] = (modeling_dataset["diff_missing_count"] == 0).astype(int)
else:
    modeling_dataset["diff_missing_count"] = np.nan
    modeling_dataset["has_all_diff_features"] = 0

core_diff_cols = [
    c for c in [
        "Diff_adj_net_rating",
        "Diff_Final_Massey",
        "Diff_Relative_Massey_Momentum",
        "Diff_SeedNum",
        "Diff_pyth_win_pct",
        "Diff_sos_adj_net_rating",
        "Diff_three_factors_composite",
        "Diff_weighted_massey_momentum",
        "Diff_avg_three_score",
        "Diff_avg_ft_score",
    ]
    if c in modeling_dataset.columns
]
if core_diff_cols:
    modeling_dataset["core_diff_missing_count"] = modeling_dataset[core_diff_cols].isna().sum(axis=1)
    modeling_dataset["has_core_diff_features"] = (modeling_dataset["core_diff_missing_count"] == 0).astype(int)
else:
    modeling_dataset["core_diff_missing_count"] = np.nan
    modeling_dataset["has_core_diff_features"] = 0

# Season-level quality profile and high-coverage filter
season_quality = (
    modeling_dataset.groupby("Season")
    .agg(
        games=("Team1Win", "size"),
        all_diff_share=("has_all_diff_features", "mean"),
        core_diff_share=("has_core_diff_features", "mean"),
        avg_diff_missing_count=("diff_missing_count", "mean"),
    )
    .reset_index()
    .sort_values("Season")
)

high_coverage_threshold = 0.80
high_coverage_seasons = set(
    season_quality.loc[season_quality["all_diff_share"] >= high_coverage_threshold, "Season"].tolist()
)

if not high_coverage_seasons:
    fallback_threshold = max(0.50, season_quality["all_diff_share"].median())
    high_coverage_seasons = set(
        season_quality.loc[season_quality["all_diff_share"] >= fallback_threshold, "Season"].tolist()
    )

modeling_dataset["is_high_coverage_season"] = modeling_dataset["Season"].isin(high_coverage_seasons).astype(int)
modeling_dataset["season_coverage_bucket"] = np.where(
    modeling_dataset["is_high_coverage_season"] == 1,
    "high",
    "low",
)

season_quality["is_high_coverage_season"] = season_quality["Season"].isin(high_coverage_seasons).astype(int)

# Export full + filtered datasets
model_out_path = modeling_path / "gam_modeling_dataset_tourney.csv"
modeling_dataset.to_csv(model_out_path, index=False)

filtered_dataset = modeling_dataset[modeling_dataset["is_high_coverage_season"] == 1].copy()
filtered_out_path = modeling_path / "gam_modeling_dataset_tourney_high_coverage.csv"
filtered_dataset.to_csv(filtered_out_path, index=False)

season_quality_out = modeling_diag_path / "modeling_dataset_quality_by_season.csv"
season_quality.to_csv(season_quality_out, index=False)

print(f"Saved tournament modeling dataset (full): {model_out_path}")
print(f"Saved tournament modeling dataset (high-coverage): {filtered_out_path}")
print(f"Saved season quality profile: {season_quality_out}")
print(f"Saved missing-target rows: {missing_target_out}")
print(f"Rows (full): {len(modeling_dataset):,} | Rows (high-coverage): {len(filtered_dataset):,}")
print("Missing target rows:", len(missing_target_rows))

if len(missing_target_rows) > 0:
    print("Preview of missing target rows:")
    display(missing_target_rows.head(20))

if "Diff_SeedNum" in modeling_dataset.columns:
    non_null_seed_diff = modeling_dataset["Diff_SeedNum"].dropna()
    if len(non_null_seed_diff) > 0:
        print(f"Diff_SeedNum min/max: {non_null_seed_diff.min():.3f} / {non_null_seed_diff.max():.3f}")

missing_seed_rate = filtered_dataset[["Team1_Seed", "Team2_Seed"]].isna().mean().mean() if "Team1_Seed" in modeling_dataset.columns else np.nan
print(f"Average seed missing rate: {missing_seed_rate:.2%}" if pd.notna(missing_seed_rate) else "Seed columns missing.")
print("High-coverage seasons:", sorted(high_coverage_seasons))

display(modeling_dataset.head(10))
display(season_quality.tail(10))

Saved top games-played discrepancies: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\top_games_played_discrepancies.csv
Saved games-played season span audit: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\games_played_season_span.csv
Top games-played discrepancy preview:
 Season  DayNum  WTeamID  LTeamID  Team1ID  Team2ID  Team1Win Team1_TeamName Team2_TeamName  Team1_games_played  Team2_games_played Team1_Seed Team2_Seed  games_played_diff  games_played_abs_diff
   2021     137     1116     1159     1116     1159         1       Arkansas        Colgate                  28                  14        Z03        Z14                 14                     14
   2006     134     1284     1214     1214     1284         0        Hampton    Monmouth NJ                  19                  31       Y16a       Y16b                -12          

,Season,DayNum,Team1ID,Team2ID,Team1Win,WTeamID,LTeamID,WScore,LScore,WLoc,...,Diff_Adj_Seed,Diff_Expected_Massey_For_Seed,Diff_Adj_Massey,Diff_AlternateTeamID,diff_missing_count,has_all_diff_features,core_diff_missing_count,has_core_diff_features,is_high_coverage_season,season_coverage_bucket
0,2005,134,1105,1324,0,1324,1105,79,69,N,...,0.0,0.252516,-10.447113,NaN,1,0,0,1,0,low
1,2005,136,1112,1429,1,1112,1429,66,53,N,...,-2.0,12.218327,-9.094909,NaN,1,0,0,1,0,low
2,2005,136,1130,1335,1,1130,1335,85,65,N,...,-4.0,10.156106,-2.345647,NaN,1,0,0,1,0,low
3,2005,136,1153,1234,1,1153,1234,76,64,N,...,1.0,0.905388,-1.196961,NaN,1,0,0,1,0,low
4,2005,136,1211,1457,1,1211,1457,74,64,N,...,-6.0,12.218327,-1.635534,NaN,1,0,0,1,0,low
5,2005,136,1228,1192,1,1228,1192,67,55,N,...,-14.0,24.984430,-0.416500,NaN,1,0,0,1,0,low
6,2005,136,1246,1184,1,1246,1184,72,64,N,...,-10.0,17.841778,0.449325,NaN,1,0,0,1,0,low
7,2005,136,1400,1305,0,1305,1400,61,57,N,...,-5.0,0.736913,6.912933,NaN,1,0,0,1,0,low
8,2005,136,1328,1310,1,1328,1310,84,67,N,...,-9.0,12.218327,4.604068,NaN,1,0,0,1,0,low
9,2005,136,1334,1338,1,1334,1338,79,71,N,...,2.0,0.736913,-3.268474,NaN,1,0,0,1,0,low


,Season,games,all_diff_share,core_diff_share,avg_diff_missing_count,is_high_coverage_season
10,2015,67,0.0,1.0,1.0,0
11,2016,67,0.0,1.0,1.0,0
12,2017,67,0.0,1.0,1.0,0
13,2018,67,0.0,1.0,1.0,0
14,2019,67,0.0,1.0,1.0,0
15,2021,66,0.0,1.0,1.0,0
16,2022,67,0.0,1.0,1.0,0
17,2023,67,0.0,1.0,1.0,0
18,2024,67,0.0,1.0,1.0,0
19,2025,67,0.0,1.0,1.0,0


In [12]:
# ---------------------------------------------------------
# Club-share exports: clean/condensed datasets for GitHub
# Unified M/W export pipeline with identical schemas by file type
# ---------------------------------------------------------
from itertools import combinations

club_share_path = CLUB_SHARE_PATH if "CLUB_SHARE_PATH" in globals() else (processed_path / "club_share")
club_share_path.mkdir(parents=True, exist_ok=True)

# Shared advanced-metric bases used in BOTH exports
# - Matchups export: Diff_<metric>
# - Team aggregates export: <metric>
shared_metric_bases = [
    "Adj_Massey", "Adj_Seed",
    "Colley_Momentum", "Final_Colley", "Final_Massey",
    "Massey_Momentum", "Relative_Colley_Momentum", "Relative_Massey_Momentum",
    "SeedNum", "SeedNumBase", "SeedPlayInFlag",
    "TournamentPrestige_5Y_Decay",
    "adj_net_rating",
    "avg_ast_rate", "avg_ast_tov_ratio", "avg_blk_pct", "avg_def_rating", "avg_drb_pct",
    "avg_efg_pct", "avg_ft_par", "avg_ft_pct", "avg_ft_score", "avg_net_rating_game",
    "avg_off_rating",
    "avg_opp_ast_rate", "avg_opp_blk_pct", "avg_opp_efg_pct", "avg_opp_massey",
    "avg_opp_stl_pct", "avg_opp_tov_pct",
    "avg_orb_pct", "avg_pf_per_poss", "avg_possessions", "avg_stl_pct",
    "avg_three_par", "avg_three_pct", "avg_three_score",
    "avg_tov_pct", "avg_trb_pct", "avg_ts_pct",
    "away_net_rating", "ceiling_score", "games_played", "home_away_net_bias", "home_net_rating",
    "net_ast_rate", "net_blk_pct", "net_efg", "net_reb", "net_stl_pct", "net_tov",
    "playmaking_defense_composite", "pyth_win_pct", "sos_adj_net_rating",
    "three_factors_composite", "weighted_massey_momentum",
]

TOURNAMENT_EXPORT_WINDOWS = {
    "M": {"historical": (2005, 2025), "team_aggregates": (2005, 2026)},
    "W": {"historical": (2014, 2025), "team_aggregates": (2014, 2026)},
}

required_diff_cols = ["Diff_Adj_Massey", "Diff_Adj_Seed"]


def first_non_null(series: pd.Series):
    non_null = series.dropna()
    if non_null.empty:
        return pd.NA
    non_blank = non_null.astype(str).str.strip()
    non_blank = non_blank[non_blank != ""]
    return non_blank.iloc[0] if not non_blank.empty else pd.NA


def first_pipe_non_null(series: pd.Series):
    non_null = series.dropna().astype(str)
    for value in non_null:
        for token in value.split("|"):
            token = token.strip()
            if token:
                return token
    return pd.NA


def resolve_existing_path(paths: list[Path], label: str) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(f"Missing required {label}. Checked: {[str(p) for p in paths]}")


def standardize_season_columns(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if "Season" not in out.columns and "season" in out.columns:
        out = out.rename(columns={"season": "Season"})
    if "Season" in out.columns:
        out["Season"] = pd.to_numeric(out["Season"], errors="coerce")
    return out


def load_tournament_inputs(t_type: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    modeling_candidates = [modeling_path / f"{t_type.lower()}_gam_modeling_dataset_tourney.csv"]
    team_candidates = [modeling_path / f"{t_type.lower()}_team_season_model_features.csv"]

    if t_type == "M":
        modeling_candidates.append(modeling_path / "gam_modeling_dataset_tourney.csv")
        team_candidates.append(modeling_path / "team_season_model_features.csv")

    if t_type == "M" and "modeling_dataset" in globals() and isinstance(modeling_dataset, pd.DataFrame):
        matchup_df = modeling_dataset.copy()
    else:
        matchup_df = pd.read_csv(resolve_existing_path(modeling_candidates, f"{t_type} modeling dataset"))

    if t_type == "M" and "team_features" in globals() and isinstance(team_features, pd.DataFrame):
        team_df = team_features.copy()
    else:
        team_df = pd.read_csv(resolve_existing_path(team_candidates, f"{t_type} team features"))

    return standardize_season_columns(matchup_df), standardize_season_columns(team_df)


def build_brand_lookup(t_type: str) -> pd.DataFrame:
    match_candidates = [name_matching_path / f"{t_type.lower()}_team_summary_to_teams_matches.csv"]
    if t_type == "M":
        match_candidates.append(name_matching_path / "team_summary_to_mteams_matches.csv")

    match_df = pd.read_csv(resolve_existing_path(match_candidates, f"{t_type} team-summary match file"))

    id_col_candidates = [f"{t_type}Teams_TeamID", "TeamID", "MTeams_TeamID", "WTeams_TeamID"]
    id_col = next((c for c in id_col_candidates if c in match_df.columns), None)
    if id_col is None:
        raise KeyError(f"No TeamID-like column found in {t_type} mapping file.")
    if "team_id" not in match_df.columns:
        raise KeyError(f"Expected 'team_id' column in {t_type} mapping file.")

    match_df = match_df[["team_id", id_col]].copy()
    match_df["team_id"] = pd.to_numeric(match_df["team_id"], errors="coerce")
    match_df["TeamID"] = pd.to_numeric(match_df[id_col], errors="coerce")
    match_df = match_df.dropna(subset=["team_id", "TeamID"]).copy()
    match_df["team_id"] = match_df["team_id"].astype(int)
    match_df["TeamID"] = match_df["TeamID"].astype(int)

    if "team_summaries" in globals() and isinstance(team_summaries, dict) and t_type in team_summaries:
        summary_df = team_summaries[t_type].copy()
    else:
        summary_path = source_path / f"{TOURNAMENT_CONFIG[t_type]['prefix']}_combined_team_box_scores_team_summary.csv"
        summary_df = pd.read_csv(summary_path)

    summary_df = summary_df.copy()
    summary_df["team_id"] = pd.to_numeric(summary_df["team_id"], errors="coerce")
    summary_df = summary_df.dropna(subset=["team_id"]).copy()
    summary_df["team_id"] = summary_df["team_id"].astype(int)

    for col in ["team_color", "team_alternate_color", "team_logo"]:
        if col not in summary_df.columns:
            summary_df[col] = pd.NA

    brand_raw = match_df.merge(
        summary_df[["team_id", "team_color", "team_alternate_color", "team_logo"]],
        on="team_id",
        how="left",
    )

    return (
        brand_raw.groupby("TeamID", as_index=False)
        .agg(
            TeamColor=("team_color", first_pipe_non_null),
            TeamAlternateColor=("team_alternate_color", first_pipe_non_null),
            TeamLogo=("team_logo", first_pipe_non_null),
        )
    )


def ensure_adj_metrics(team_df: pd.DataFrame) -> pd.DataFrame:
    out = team_df.copy()

    if "SeedNum" not in out.columns:
        if "Seed" in out.columns:
            out["SeedNum"] = pd.to_numeric(out["Seed"].astype(str).str.extract(r"(\d{2})", expand=False), errors="coerce")
        else:
            out["SeedNum"] = np.nan
    else:
        out["SeedNum"] = pd.to_numeric(out["SeedNum"], errors="coerce")

    massey_col = "Final_Massey" if "Final_Massey" in out.columns else ("Final_Massey_Rating" if "Final_Massey_Rating" in out.columns else None)

    if "Adj_Seed" not in out.columns:
        out["Adj_Seed"] = np.nan
    if "Expected_Seed" not in out.columns:
        out["Expected_Seed"] = np.nan

    if massey_col is not None and "SeedNum" in out.columns:
        seed_fit = out[[massey_col, "SeedNum"]].dropna().copy()
        if len(seed_fit) >= 2:
            slope_seed, intercept_seed = np.polyfit(seed_fit[massey_col], seed_fit["SeedNum"], 1)
            expected_seed = intercept_seed + slope_seed * pd.to_numeric(out[massey_col], errors="coerce")
            expected_seed = np.clip(expected_seed, 1, 16)
            out["Expected_Seed"] = out["Expected_Seed"].where(out["Expected_Seed"].notna(), expected_seed)
            out["Adj_Seed"] = out["Adj_Seed"].where(out["Adj_Seed"].notna(), np.round(expected_seed))

    if "Adj_Massey" not in out.columns:
        out["Adj_Massey"] = np.nan
    if "Expected_Massey_For_Seed" not in out.columns:
        out["Expected_Massey_For_Seed"] = np.nan

    if massey_col is not None and "SeedNum" in out.columns:
        seed_fit = out[[massey_col, "SeedNum"]].dropna().copy()
        if not seed_fit.empty:
            expected_massey_by_seed = seed_fit.groupby("SeedNum")[massey_col].mean()
            inferred = out["SeedNum"].map(expected_massey_by_seed)
            out["Expected_Massey_For_Seed"] = out["Expected_Massey_For_Seed"].where(
                out["Expected_Massey_For_Seed"].notna(),
                inferred,
            )
            out["Adj_Massey"] = out["Adj_Massey"].where(
                out["Adj_Massey"].notna(),
                pd.to_numeric(out[massey_col], errors="coerce") - pd.to_numeric(out["Expected_Massey_For_Seed"], errors="coerce"),
            )

    return out


def backfill_required_diffs(matchups: pd.DataFrame, team_agg: pd.DataFrame) -> pd.DataFrame:
    out = matchups.copy()

    lookup_cols = ["Season", "TeamID", "Adj_Massey", "Adj_Seed"]
    for col in lookup_cols:
        if col not in team_agg.columns:
            team_agg[col] = np.nan

    lookup = team_agg[lookup_cols].drop_duplicates(subset=["Season", "TeamID"])

    t1 = lookup.rename(columns={
        "TeamID": "Team1ID",
        "Adj_Massey": "Team1_Adj_Massey",
        "Adj_Seed": "Team1_Adj_Seed",
    })
    t2 = lookup.rename(columns={
        "TeamID": "Team2ID",
        "Adj_Massey": "Team2_Adj_Massey",
        "Adj_Seed": "Team2_Adj_Seed",
    })

    out = out.merge(t1, on=["Season", "Team1ID"], how="left")
    out = out.merge(t2, on=["Season", "Team2ID"], how="left")

    for metric in ["Adj_Massey", "Adj_Seed"]:
        diff_col = f"Diff_{metric}"
        calc = pd.to_numeric(out[f"Team1_{metric}"], errors="coerce") - pd.to_numeric(out[f"Team2_{metric}"], errors="coerce")
        if diff_col not in out.columns:
            out[diff_col] = calc
        else:
            out[diff_col] = out[diff_col].where(out[diff_col].notna(), calc)

    out = out.drop(columns=["Team1_Adj_Massey", "Team1_Adj_Seed", "Team2_Adj_Massey", "Team2_Adj_Seed"], errors="ignore")
    return out


# Remove legacy unprefixed/parquet club-share artifacts so only M/W CSVs are produced by this cell.
legacy_artifacts = [
    "historical_matchups_2005_2025.csv",
    "team_aggregates_2005_2026.csv",
    "modeling_matchups_2026_all_possible.csv",
    "historical_matchups_2005_2025.parquet",
    "team_aggregates_2005_2026.parquet",
    "modeling_matchups_2026_all_possible.parquet",
]
for legacy_name in legacy_artifacts:
    legacy_path = club_share_path / legacy_name
    if legacy_path.exists():
        legacy_path.unlink()


dual_pipeline_outputs = {}
for t_type in TOURNAMENT_TYPES:
    windows = TOURNAMENT_EXPORT_WINDOWS[t_type]
    hist_start, hist_end = windows["historical"]
    agg_start, agg_end = windows["team_aggregates"]

    matchup_df, team_df = load_tournament_inputs(t_type)
    team_df = ensure_adj_metrics(team_df)

    brand_lookup = build_brand_lookup(t_type)
    team1_brand = brand_lookup.rename(columns={
        "TeamID": "Team1ID",
        "TeamColor": "Team1_Color",
        "TeamAlternateColor": "Team1_AlternateColor",
        "TeamLogo": "Team1_Logo",
    })
    team2_brand = brand_lookup.rename(columns={
        "TeamID": "Team2ID",
        "TeamColor": "Team2_Color",
        "TeamAlternateColor": "Team2_AlternateColor",
        "TeamLogo": "Team2_Logo",
    })

    matchup_df = standardize_season_columns(matchup_df)
    team_df = standardize_season_columns(team_df)

    if "Season" not in matchup_df.columns or "Season" not in team_df.columns:
        raise KeyError(f"Season column missing for tournament {t_type}.")

    matchup_df = matchup_df[matchup_df["Season"].between(hist_start, season_end)].copy()
    team_agg = team_df[team_df["Season"].between(agg_start, agg_end)].copy()

    for id_col in ["TeamID"]:
        team_agg[id_col] = pd.to_numeric(team_agg[id_col], errors="coerce")
    team_agg = team_agg[team_agg["TeamID"].notna()].copy()
    team_agg["TeamID"] = team_agg["TeamID"].astype(int)

    # Ensure identical team-aggregate metric schema for M/W exports.
    for metric in shared_metric_bases:
        if metric not in team_agg.columns:
            team_agg[metric] = np.nan

    team_agg = team_agg.merge(brand_lookup, on="TeamID", how="left")

    team_ordered_cols = [
        "Season", "TeamID", "TeamName", "Seed",
        *shared_metric_bases,
        "TeamColor", "TeamAlternateColor", "TeamLogo",
    ]
    for col in team_ordered_cols:
        if col not in team_agg.columns:
            team_agg[col] = np.nan
    team_agg_clean = team_agg[team_ordered_cols].copy()

    matchups_hist = matchup_df[matchup_df["Season"].between(hist_start, hist_end)].copy()
    if "Team1Win" in matchups_hist.columns:
        matchups_hist = matchups_hist[matchups_hist["Team1Win"].notna()].copy()
        matchups_hist["Team1Win"] = pd.to_numeric(matchups_hist["Team1Win"], errors="coerce").astype("Int64")
        if "Target_Team1Win" not in matchups_hist.columns:
            matchups_hist["Target_Team1Win"] = matchups_hist["Team1Win"]
    elif "Target_Team1Win" not in matchups_hist.columns:
        raise KeyError(f"Historical matchup target is missing for tournament {t_type}.")

    matchups_hist = backfill_required_diffs(matchups_hist, team_agg_clean)

    historical_base_cols = [
        "Season", "DayNum", "Team1ID", "Team2ID",
        "Team1_TeamName", "Team2_TeamName",
        "Team1_Seed", "Team2_Seed",
        "Diff_SeedNum", "Target_Team1Win",
    ]
    historical_diff_cols = [
        f"Diff_{metric}" for metric in shared_metric_bases
        if f"Diff_{metric}" != "Diff_SeedNum"
    ]
    historical_brand_cols = [
        "Team1_Color", "Team1_AlternateColor", "Team1_Logo",
        "Team2_Color", "Team2_AlternateColor", "Team2_Logo",
    ]

    for col in historical_base_cols + historical_diff_cols:
        if col not in matchups_hist.columns:
            matchups_hist[col] = np.nan

    matchups_hist_clean = matchups_hist[historical_base_cols + historical_diff_cols].copy()
    matchups_hist_clean = (
        matchups_hist_clean
        .merge(team1_brand, on="Team1ID", how="left")
        .merge(team2_brand, on="Team2ID", how="left")
    )

    for col in historical_brand_cols:
        if col not in matchups_hist_clean.columns:
            matchups_hist_clean[col] = np.nan
    matchups_hist_clean = matchups_hist_clean[historical_base_cols + historical_diff_cols + historical_brand_cols]

    # Build 2026 all-possible seeded matchups using the same metric contract.
    season_2026 = 2026
    team_agg_2026 = team_agg_clean[team_agg_clean["Season"] == season_2026].copy()

    if "SeedNum" in team_agg_2026.columns:
        team_agg_2026["SeedNum"] = pd.to_numeric(team_agg_2026["SeedNum"], errors="coerce")
    else:
        team_agg_2026["SeedNum"] = pd.to_numeric(team_agg_2026["Seed"].astype(str).str.extract(r"(\d{2})", expand=False), errors="coerce")

    official_seed_mask = team_agg_2026["Seed"].astype(str).str.match(r"^[WXYZ]\d{2}[ab]?$", na=False)
    team_agg_2026 = team_agg_2026[official_seed_mask & team_agg_2026["SeedNum"].notna()].copy()
    if team_agg_2026.empty:
        raise RuntimeError(f"No official 2026 seeded rows found for {t_type} all-possible export.")

    massey_sort_col = "Final_Massey" if "Final_Massey" in team_agg_2026.columns else None
    if massey_sort_col is not None:
        team_agg_2026[massey_sort_col] = pd.to_numeric(team_agg_2026[massey_sort_col], errors="coerce")

    team_agg_2026 = team_agg_2026.drop_duplicates(subset=["TeamID"]).copy()
    team_agg_2026 = team_agg_2026.sort_values(["SeedNum", "TeamID"])

    records_2026 = []
    team_ids_2026 = team_agg_2026["TeamID"].astype(int).tolist()
    for team_a_id, team_b_id in combinations(team_ids_2026, 2):
        row_a = team_agg_2026.loc[team_agg_2026["TeamID"] == team_a_id].iloc[0]
        row_b = team_agg_2026.loc[team_agg_2026["TeamID"] == team_b_id].iloc[0]

        team1_is_a = True
        if pd.notna(row_a["SeedNum"]) and pd.notna(row_b["SeedNum"]) and row_a["SeedNum"] != row_b["SeedNum"]:
            team1_is_a = row_a["SeedNum"] < row_b["SeedNum"]
        elif massey_sort_col is not None and pd.notna(row_a[massey_sort_col]) and pd.notna(row_b[massey_sort_col]) and row_a[massey_sort_col] != row_b[massey_sort_col]:
            team1_is_a = row_a[massey_sort_col] > row_b[massey_sort_col]
        else:
            team1_is_a = int(team_a_id) < int(team_b_id)

        row_t1 = row_a if team1_is_a else row_b
        row_t2 = row_b if team1_is_a else row_a

        rec = {
            "Season": season_2026,
            "Team1ID": int(row_t1["TeamID"]),
            "Team2ID": int(row_t2["TeamID"]),
            "Team1_TeamName": row_t1.get("TeamName", pd.NA),
            "Team2_TeamName": row_t2.get("TeamName", pd.NA),
            "Team1_Seed": row_t1.get("Seed", pd.NA),
            "Team2_Seed": row_t2.get("Seed", pd.NA),
            "Team1_SeedRegion": str(row_t1.get("Seed", ""))[:1],
            "Team2_SeedRegion": str(row_t2.get("Seed", ""))[:1],
        }

        for metric in shared_metric_bases:
            v1 = pd.to_numeric(row_t1.get(metric, np.nan), errors="coerce")
            v2 = pd.to_numeric(row_t2.get(metric, np.nan), errors="coerce")
            rec[f"Diff_{metric}"] = v1 - v2 if pd.notna(v1) and pd.notna(v2) else np.nan

        if pd.notna(row_t1.get("SeedNum", np.nan)) and pd.notna(row_t2.get("SeedNum", np.nan)):
            rec["Diff_SeedNum"] = float(row_t2["SeedNum"]) - float(row_t1["SeedNum"])
        else:
            rec["Diff_SeedNum"] = np.nan

        records_2026.append(rec)

    matchups_2026_all = pd.DataFrame(records_2026)

    all_possible_base_cols = [
        "Season", "Team1ID", "Team2ID",
        "Team1_TeamName", "Team2_TeamName",
        "Team1_Seed", "Team2_Seed",
        "Team1_SeedRegion", "Team2_SeedRegion",
        "Diff_SeedNum",
    ]
    all_possible_diff_cols = [
        f"Diff_{metric}" for metric in shared_metric_bases
        if f"Diff_{metric}" != "Diff_SeedNum"
    ]
    all_possible_brand_cols = [
        "Team1_Color", "Team1_AlternateColor", "Team1_Logo",
        "Team2_Color", "Team2_AlternateColor", "Team2_Logo",
    ]

    for col in all_possible_base_cols + all_possible_diff_cols:
        if col not in matchups_2026_all.columns:
            matchups_2026_all[col] = np.nan

    matchups_2026_all = (
        matchups_2026_all[all_possible_base_cols + all_possible_diff_cols]
        .merge(team1_brand, on="Team1ID", how="left")
        .merge(team2_brand, on="Team2ID", how="left")
    )

    for col in all_possible_brand_cols:
        if col not in matchups_2026_all.columns:
            matchups_2026_all[col] = np.nan
    matchups_2026_all = matchups_2026_all[all_possible_base_cols + all_possible_diff_cols + all_possible_brand_cols]

    for req_col in required_diff_cols:
        if req_col not in matchups_hist_clean.columns or req_col not in matchups_2026_all.columns:
            raise RuntimeError(f"Missing required column {req_col} for tournament {t_type}.")

    hist_out = club_share_path / CLUB_SHARE_EXPORT_NAMES[t_type]["historical"]
    agg_out = club_share_path / CLUB_SHARE_EXPORT_NAMES[t_type]["team_aggregates"]
    all_out = club_share_path / CLUB_SHARE_EXPORT_NAMES[t_type]["all_possible_2026"]

    matchups_hist_clean.to_csv(hist_out, index=False)
    team_agg_clean.to_csv(agg_out, index=False)
    matchups_2026_all.to_csv(all_out, index=False)

    dual_pipeline_outputs[t_type] = {
        "historical": matchups_hist_clean,
        "team_aggregates": team_agg_clean,
        "all_possible_2026": matchups_2026_all,
        "historical_path": hist_out,
        "team_aggregates_path": agg_out,
        "all_possible_2026_path": all_out,
    }

    print(f"[{t_type}] Historical matchups saved: {hist_out}")
    print(f"[{t_type}] Team aggregates saved:   {agg_out}")
    print(f"[{t_type}] 2026 all-possible saved: {all_out}")
    print(f"[{t_type}] Rows - historical: {len(matchups_hist_clean):,} | team agg: {len(team_agg_clean):,} | all-possible: {len(matchups_2026_all):,}")
    print(f"[{t_type}] Required diff non-null share: Diff_Adj_Massey={matchups_hist_clean['Diff_Adj_Massey'].notna().mean():.1%} | Diff_Adj_Seed={matchups_hist_clean['Diff_Adj_Seed'].notna().mean():.1%}")

# Promote men outputs as defaults for existing downstream diagnostics cells that expect these names.
matchups_2026_all = dual_pipeline_outputs["M"]["all_possible_2026"].copy()
team_agg_clean = dual_pipeline_outputs["M"]["team_aggregates"].copy()

# Schema parity check across M/W exports.
for export_key in ["historical", "team_aggregates", "all_possible_2026"]:
    m_cols = dual_pipeline_outputs["M"][export_key].columns.tolist()
    w_cols = dual_pipeline_outputs["W"][export_key].columns.tolist()
    if m_cols != w_cols:
        raise AssertionError(f"Schema mismatch for {export_key}: M/W column order differs.")

print("Schema parity check passed: M/W club-share exports are column-identical by file type.")

display(dual_pipeline_outputs["M"]["historical"].head(5))
display(dual_pipeline_outputs["W"]["historical"].head(5))
display(dual_pipeline_outputs["M"]["all_possible_2026"].head(5))
display(dual_pipeline_outputs["W"]["all_possible_2026"].head(5))

[M] Historical matchups saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\m_historical_matchups_2005_2025.csv
[M] Team aggregates saved:   x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\m_team_aggregates_2005_2026.csv
[M] 2026 all-possible saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\m_modeling_matchups_2026_all_possible.csv
[M] Rows - historical: 1,321 | team agg: 7,798 | all-possible: 2,016
[M] Required diff non-null share: Diff_Adj_Massey=100.0% | Diff_Adj_Seed=100.0%
[W] Historical matchups saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\w_historical_matchups_2014_2025.csv
[W] Team aggregates saved:   x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\w_team_aggregates_2014_2026.csv

,Season,DayNum,Team1ID,Team2ID,Team1_TeamName,Team2_TeamName,Team1_Seed,Team2_Seed,Diff_SeedNum,Target_Team1Win,...,Diff_pyth_win_pct,Diff_sos_adj_net_rating,Diff_three_factors_composite,Diff_weighted_massey_momentum,Team1_Color,Team1_AlternateColor,Team1_Logo,Team2_Color,Team2_AlternateColor,Team2_Logo
0,2005,134,1105,1324,Alabama A&M,Oakland,Z16a,Z16b,0.5,0,...,0.059791,-8.053033,0.211558,-2.046903,790000,ffffff,https://a.espncdn.com/i/teamlogos/ncaa/500/201...,04091c,bc955c,https://a.espncdn.com/i/teamlogos/ncaa/500/247...
1,2005,136,1112,1429,Arizona,Utah St,X03,X14,11.0,1,...,-0.117182,-4.612147,-0.408777,0.173382,0c234b,003366,https://a.espncdn.com/i/teamlogos/ncaa/500/12.png,00263a,949ca1,https://a.espncdn.com/i/teamlogos/ncaa/500/328...
2,2005,136,1130,1335,Boston College,Penn,X04,X13,9.0,1,...,0.056762,18.831034,0.071363,0.026081,8c2232,dbcca6,https://a.espncdn.com/i/teamlogos/ncaa/500/103...,082A74,a6163d,https://a.espncdn.com/i/teamlogos/ncaa/500/219...
3,2005,136,1153,1234,Cincinnati,Iowa,Y07,Y10,3.0,1,...,0.099584,0.139939,0.222970,-0.420335,000000,717073,https://a.espncdn.com/i/teamlogos/ncaa/500/213...,000000,fcd116,https://a.espncdn.com/i/teamlogos/ncaa/500/229...
4,2005,136,1211,1457,Gonzaga,Winthrop,W03,W14,11.0,1,...,0.024373,12.366357,-0.114872,-0.517654,041e42,c8102e,https://a.espncdn.com/i/teamlogos/ncaa/500/225...,9e0b0e,fdb41e,https://a.espncdn.com/i/teamlogos/ncaa/500/273...


,Season,DayNum,Team1ID,Team2ID,Team1_TeamName,Team2_TeamName,Team1_Seed,Team2_Seed,Diff_SeedNum,Target_Team1Win,...,Diff_pyth_win_pct,Diff_sos_adj_net_rating,Diff_three_factors_composite,Diff_weighted_massey_momentum,Team1_Color,Team1_AlternateColor,Team1_Logo,Team2_Color,Team2_AlternateColor,Team2_Logo
0,2014,138,3435,3113,Vanderbilt,Arizona State,Z08,Z09,1.0,0,...,-0.026606,0.902415,-0.111135,-0.320400,000000,231f20,https://a.espncdn.com/i/teamlogos/ncaa/500/238...,8e0c3a,8c1d40,https://a.espncdn.com/i/teamlogos/ncaa/500/9.png
1,2014,138,3124,3443,Baylor,Western KY,Z02,Z15,13.0,1,...,0.654972,38.861718,1.369402,1.044316,154734,ffb81c,https://a.espncdn.com/i/teamlogos/ncaa/500/239...,F32026,b3b5b8,https://a.espncdn.com/i/teamlogos/ncaa/500/98.png
2,2014,138,3301,3140,NC State,BYU,W05,W12,7.0,0,...,0.116481,12.754295,0.622696,-4.094763,EF1216,231f20,https://a.espncdn.com/i/teamlogos/ncaa/500/152...,001E4C,002e5d,https://a.espncdn.com/i/teamlogos/ncaa/500/252...
3,2014,138,3143,3200,California,Fordham,Z07,Z10,3.0,1,...,-0.219854,-15.056043,-0.155071,-2.083356,031522,ffc423,https://a.espncdn.com/i/teamlogos/ncaa/500/25.png,830032,909090,https://a.espncdn.com/i/teamlogos/ncaa/500/223...
4,2014,138,3177,3328,DePaul,Oklahoma,W07,W10,3.0,1,...,-0.089223,0.620487,-0.060670,-0.021127,2d649c,ce1125,https://a.espncdn.com/i/teamlogos/ncaa/500/305...,990000,ffffff,https://a.espncdn.com/i/teamlogos/ncaa/500/201...


,Season,Team1ID,Team2ID,Team1_TeamName,Team2_TeamName,Team1_Seed,Team2_Seed,Team1_SeedRegion,Team2_SeedRegion,Diff_SeedNum,...,Diff_pyth_win_pct,Diff_sos_adj_net_rating,Diff_three_factors_composite,Diff_weighted_massey_momentum,Team1_Color,Team1_AlternateColor,Team1_Logo,Team2_Color,Team2_AlternateColor,Team2_Logo
0,2026,1181,1112,Duke,Arizona,W01,Z01,W,Z,0.0,...,0.024723,12.779964,0.247467,0.233461,00539b,ffffff,https://a.espncdn.com/i/teamlogos/ncaa/500/150...,0c234b,003366,https://a.espncdn.com/i/teamlogos/ncaa/500/12.png
1,2026,1112,1196,Arizona,Florida,Z01,X01,Z,X,0.0,...,0.034980,-1.999963,0.123857,-0.330847,0c234b,003366,https://a.espncdn.com/i/teamlogos/ncaa/500/12.png,0021a5,fa4616,https://a.espncdn.com/i/teamlogos/ncaa/500/57.png
2,2026,1276,1112,Michigan,Arizona,Y01,Z01,Y,Z,0.0,...,0.005670,2.471907,-0.198232,0.006314,00274c,ffcb05,https://a.espncdn.com/i/teamlogos/ncaa/500/130...,0c234b,003366,https://a.espncdn.com/i/teamlogos/ncaa/500/12.png
3,2026,1112,1163,Arizona,Connecticut,Z01,W02,Z,W,1.0,...,0.046549,5.990973,0.255008,0.185785,0c234b,003366,https://a.espncdn.com/i/teamlogos/ncaa/500/12.png,0c2340,a2aaad,https://a.espncdn.com/i/teamlogos/ncaa/500/41.png
4,2026,1112,1222,Arizona,Houston,Z01,X02,Z,X,1.0,...,0.014678,4.827647,-0.044321,-0.038515,0c234b,003366,https://a.espncdn.com/i/teamlogos/ncaa/500/12.png,c8102e,ffffff,https://a.espncdn.com/i/teamlogos/ncaa/500/248...


,Season,Team1ID,Team2ID,Team1_TeamName,Team2_TeamName,Team1_Seed,Team2_Seed,Team1_SeedRegion,Team2_SeedRegion,Diff_SeedNum,...,Diff_pyth_win_pct,Diff_sos_adj_net_rating,Diff_three_factors_composite,Diff_weighted_massey_momentum,Team1_Color,Team1_AlternateColor,Team1_Logo,Team2_Color,Team2_AlternateColor,Team2_Logo
0,2026,3163,3376,UConn,South Carolina,W01,Z01,W,Z,0.0,...,0.005390,16.759618,0.595653,-0.332510,0c2340,a2aaad,https://a.espncdn.com/i/teamlogos/ncaa/500/41.png,73000a,000000,https://a.espncdn.com/i/teamlogos/ncaa/500/257...
1,2026,3163,3400,UConn,Texas,W01,Y01,W,Y,0.0,...,0.007480,25.625910,0.238244,-0.068036,0c2340,a2aaad,https://a.espncdn.com/i/teamlogos/ncaa/500/41.png,af5c37,ffffff,https://a.espncdn.com/i/teamlogos/ncaa/500/251...
2,2026,3163,3417,UConn,UCLA,W01,X01,W,X,0.0,...,0.007710,16.117176,0.546815,-0.144036,0c2340,a2aaad,https://a.espncdn.com/i/teamlogos/ncaa/500/41.png,2774ae,f2a900,https://a.espncdn.com/i/teamlogos/ncaa/500/26.png
3,2026,3163,3181,UConn,Duke,W01,Y02,W,Y,1.0,...,0.060804,37.590839,1.319479,-0.251725,0c2340,a2aaad,https://a.espncdn.com/i/teamlogos/ncaa/500/41.png,00539b,ffffff,https://a.espncdn.com/i/teamlogos/ncaa/500/150...
4,2026,3163,3234,UConn,Iowa,W01,Z02,W,Z,1.0,...,0.101473,33.855342,1.449414,-0.222646,0c2340,a2aaad,https://a.espncdn.com/i/teamlogos/ncaa/500/41.png,000000,fcd116,https://a.espncdn.com/i/teamlogos/ncaa/500/229...


In [13]:
# --------------------------------------------------------------------------------------
# 68-team rebuild for 2026 all-possible exports using TeamID + AlternateTeamID seed slots
# --------------------------------------------------------------------------------------
from itertools import combinations

season_68 = 2026
official_seed_pattern = r"^[WXYZ]\d{2}[ab]?$"
seed_sheet_by_tournament = {
    "M": "MNCAATourneySeeds",
    "W": "WNCAATourneySeeds",
}
seed_file_candidates = [
    club_share_path / "2026_Seeding.xlsx",
    source_path / "2026_Seeding.xlsx",
]
seed_file_path = next((p for p in seed_file_candidates if p.exists()), None)

if seed_file_path is None:
    raise FileNotFoundError("2026_Seeding.xlsx not found. Cannot build 68-team all-possible exports.")


def load_seed_slots_2026(sheet_name: str) -> pd.DataFrame:
    slots = pd.read_excel(seed_file_path, sheet_name=sheet_name)
    required_cols = {"Season", "Seed", "TeamID"}
    missing = required_cols - set(slots.columns)
    if missing:
        raise KeyError(f"Missing required seeding columns in {sheet_name}: {sorted(missing)}")

    if "AlternateTeamID" not in slots.columns:
        slots["AlternateTeamID"] = np.nan

    slots = slots[["Season", "Seed", "TeamID", "AlternateTeamID"]].copy()
    slots["Season"] = pd.to_numeric(slots["Season"], errors="coerce")
    slots["TeamID"] = pd.to_numeric(slots["TeamID"], errors="coerce")
    slots["AlternateTeamID"] = pd.to_numeric(slots["AlternateTeamID"], errors="coerce")
    slots["Seed"] = slots["Seed"].astype(str).str.strip()
    slots = slots.dropna(subset=["Season", "Seed", "TeamID"]).copy()
    slots["Season"] = slots["Season"].astype(int)
    slots["TeamID"] = slots["TeamID"].astype(int)
    slots = slots[slots["Season"] == season_68].copy()
    slots = slots[slots["Seed"].str.match(official_seed_pattern, na=False)].copy()
    slots = slots.drop_duplicates(subset=["Season", "Seed"], keep="last").copy()
    return slots


def build_field_68(seed_slots: pd.DataFrame) -> pd.DataFrame:
    primary = seed_slots[["Season", "Seed", "TeamID"]].copy()
    primary["SeedSlotRole"] = "primary"

    alternate = (
        seed_slots[["Season", "Seed", "AlternateTeamID"]]
        .rename(columns={"AlternateTeamID": "TeamID"})
        .dropna(subset=["TeamID"])
        .copy()
    )
    if not alternate.empty:
        alternate["TeamID"] = alternate["TeamID"].astype(int)
    alternate["SeedSlotRole"] = "alternate"

    field = pd.concat([primary, alternate], ignore_index=True)
    field = field.drop_duplicates(subset=["Season", "Seed", "TeamID"]).copy()
    return field


field_68_by_tournament = {}
for t_type in TOURNAMENT_TYPES:
    team_agg_2026 = dual_pipeline_outputs[t_type]["team_aggregates"].copy()
    team_agg_2026["Season"] = pd.to_numeric(team_agg_2026["Season"], errors="coerce")
    team_agg_2026 = team_agg_2026[team_agg_2026["Season"] == season_68].copy()
    team_agg_2026["TeamID"] = pd.to_numeric(team_agg_2026["TeamID"], errors="coerce")
    team_agg_2026 = team_agg_2026.dropna(subset=["TeamID"]).copy()
    team_agg_2026["TeamID"] = team_agg_2026["TeamID"].astype(int)

    if "SeedNum" not in team_agg_2026.columns:
        team_agg_2026["SeedNum"] = pd.to_numeric(
            team_agg_2026["Seed"].astype(str).str.extract(r"(\d{2})", expand=False),
            errors="coerce",
        )
    else:
        team_agg_2026["SeedNum"] = pd.to_numeric(team_agg_2026["SeedNum"], errors="coerce")

    if "SeedNumBase" not in team_agg_2026.columns:
        team_agg_2026["SeedNumBase"] = pd.to_numeric(
            team_agg_2026["Seed"].astype(str).str.extract(r"(\d{2})", expand=False),
            errors="coerce",
        )

    seed_slots_2026 = load_seed_slots_2026(seed_sheet_by_tournament[t_type])
    field_68 = build_field_68(seed_slots_2026)

    field_68 = field_68.merge(team_agg_2026.drop(columns=["Seed"], errors="ignore"), on=["Season", "TeamID"], how="left")
    field_68["SeedNum"] = pd.to_numeric(field_68.get("SeedNum", np.nan), errors="coerce")
    field_68["SeedNum"] = field_68["SeedNum"].where(field_68["SeedNum"].notna(), pd.to_numeric(field_68["Seed"].str.extract(r"(\d{2})", expand=False), errors="coerce"))
    field_68["SeedNumBase"] = pd.to_numeric(field_68.get("SeedNumBase", np.nan), errors="coerce")
    field_68["SeedNumBase"] = field_68["SeedNumBase"].where(field_68["SeedNumBase"].notna(), pd.to_numeric(field_68["Seed"].str.extract(r"(\d{2})", expand=False), errors="coerce"))
    field_68["SeedPlayInFlag"] = field_68["Seed"].astype(str).str.contains(r"[ab]$", regex=True, na=False).astype(int)
    field_68["SeedRegion"] = field_68["Seed"].astype(str).str[0]

    role_order = {"primary": 0, "alternate": 1}
    field_68["_role_sort"] = field_68["SeedSlotRole"].map(role_order).fillna(99)
    field_68 = field_68.sort_values(["_role_sort", "SeedNum", "TeamID"], na_position="last")
    field_68 = field_68.drop_duplicates(subset=["TeamID"], keep="first").drop(columns=["_role_sort"])  # safety for duplicate IDs

    massey_sort_col = "Final_Massey" if "Final_Massey" in field_68.columns else None
    if massey_sort_col is not None:
        field_68[massey_sort_col] = pd.to_numeric(field_68[massey_sort_col], errors="coerce")

    team_ids_2026 = field_68["TeamID"].astype(int).tolist()
    records_2026 = []
    for team_a_id, team_b_id in combinations(team_ids_2026, 2):
        row_a = field_68.loc[field_68["TeamID"] == team_a_id].iloc[0]
        row_b = field_68.loc[field_68["TeamID"] == team_b_id].iloc[0]

        team1_is_a = True
        if pd.notna(row_a["SeedNum"]) and pd.notna(row_b["SeedNum"]) and row_a["SeedNum"] != row_b["SeedNum"]:
            team1_is_a = row_a["SeedNum"] < row_b["SeedNum"]
        elif massey_sort_col is not None and pd.notna(row_a[massey_sort_col]) and pd.notna(row_b[massey_sort_col]) and row_a[massey_sort_col] != row_b[massey_sort_col]:
            team1_is_a = row_a[massey_sort_col] > row_b[massey_sort_col]
        else:
            team1_is_a = int(team_a_id) < int(team_b_id)

        row_t1 = row_a if team1_is_a else row_b
        row_t2 = row_b if team1_is_a else row_a

        rec = {
            "Season": season_68,
            "Team1ID": int(row_t1["TeamID"]),
            "Team2ID": int(row_t2["TeamID"]),
            "Team1_TeamName": row_t1.get("TeamName", pd.NA),
            "Team2_TeamName": row_t2.get("TeamName", pd.NA),
            "Team1_Seed": row_t1.get("Seed", pd.NA),
            "Team2_Seed": row_t2.get("Seed", pd.NA),
            "Team1_SeedRegion": str(row_t1.get("Seed", ""))[:1],
            "Team2_SeedRegion": str(row_t2.get("Seed", ""))[:1],
            "Team1_Field68Role": row_t1.get("SeedSlotRole", pd.NA),
            "Team2_Field68Role": row_t2.get("SeedSlotRole", pd.NA),
        }

        for metric in shared_metric_bases:
            v1 = pd.to_numeric(row_t1.get(metric, np.nan), errors="coerce")
            v2 = pd.to_numeric(row_t2.get(metric, np.nan), errors="coerce")
            rec[f"Diff_{metric}"] = v1 - v2 if pd.notna(v1) and pd.notna(v2) else np.nan

        if pd.notna(row_t1.get("SeedNum", np.nan)) and pd.notna(row_t2.get("SeedNum", np.nan)):
            rec["Diff_SeedNum"] = float(row_t2["SeedNum"]) - float(row_t1["SeedNum"])
        else:
            rec["Diff_SeedNum"] = np.nan

        records_2026.append(rec)

    matchups_2026_all = pd.DataFrame(records_2026)

    all_possible_base_cols = [
        "Season", "Team1ID", "Team2ID",
        "Team1_TeamName", "Team2_TeamName",
        "Team1_Seed", "Team2_Seed",
        "Team1_SeedRegion", "Team2_SeedRegion",
        "Team1_Field68Role", "Team2_Field68Role",
        "Diff_SeedNum",
    ]
    all_possible_diff_cols = [
        f"Diff_{metric}" for metric in shared_metric_bases
        if f"Diff_{metric}" != "Diff_SeedNum"
    ]
    all_possible_brand_cols = [
        "Team1_Color", "Team1_AlternateColor", "Team1_Logo",
        "Team2_Color", "Team2_AlternateColor", "Team2_Logo",
    ]

    for col in all_possible_base_cols + all_possible_diff_cols:
        if col not in matchups_2026_all.columns:
            matchups_2026_all[col] = np.nan

    brand_lookup = team_agg_2026[["TeamID", "TeamColor", "TeamAlternateColor", "TeamLogo"]].drop_duplicates("TeamID")
    team1_brand = brand_lookup.rename(columns={
        "TeamID": "Team1ID",
        "TeamColor": "Team1_Color",
        "TeamAlternateColor": "Team1_AlternateColor",
        "TeamLogo": "Team1_Logo",
    })
    team2_brand = brand_lookup.rename(columns={
        "TeamID": "Team2ID",
        "TeamColor": "Team2_Color",
        "TeamAlternateColor": "Team2_AlternateColor",
        "TeamLogo": "Team2_Logo",
    })

    matchups_2026_all = (
        matchups_2026_all[all_possible_base_cols + all_possible_diff_cols]
        .merge(team1_brand, on="Team1ID", how="left")
        .merge(team2_brand, on="Team2ID", how="left")
    )

    for col in all_possible_brand_cols:
        if col not in matchups_2026_all.columns:
            matchups_2026_all[col] = np.nan
    matchups_2026_all = matchups_2026_all[all_possible_base_cols + all_possible_diff_cols + all_possible_brand_cols]

    out_path = dual_pipeline_outputs[t_type]["all_possible_2026_path"]
    matchups_2026_all.to_csv(out_path, index=False)

    dual_pipeline_outputs[t_type]["all_possible_2026"] = matchups_2026_all
    dual_pipeline_outputs[t_type]["field_68"] = field_68
    field_68_by_tournament[t_type] = field_68

    expected_pairs = len(field_68) * (len(field_68) - 1) // 2
    print(f"[{t_type}] 68-team rebuild saved: {out_path}")
    print(f"[{t_type}] field_68 teams: {len(field_68):,} | expected all-pairs: {expected_pairs:,} | actual: {len(matchups_2026_all):,}")

# Keep men defaults for downstream diagnostics that expect these names.
matchups_2026_all = dual_pipeline_outputs["M"]["all_possible_2026"].copy()
team_agg_clean = dual_pipeline_outputs["M"]["team_aggregates"].copy()

[M] 68-team rebuild saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\m_modeling_matchups_2026_all_possible.csv
[M] field_68 teams: 68 | expected all-pairs: 2,278 | actual: 2,278
[W] 68-team rebuild saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\w_modeling_matchups_2026_all_possible.csv
[W] field_68 teams: 68 | expected all-pairs: 2,278 | actual: 2,278


In [14]:
# ------------------------------------------------------------------------
# Team-aggregate seed enrichment for 2026 alternate seed-slot teams (M/W)
# Run this after the 68-team rebuild cell and before first-round export.
# ------------------------------------------------------------------------
season_seed_fix = 2026
official_seed_pattern = r"^[WXYZ]\d{2}[ab]?$"
seed_sheet_by_tournament = {
    "M": "MNCAATourneySeeds",
    "W": "WNCAATourneySeeds",
}
seed_file_candidates = [
    club_share_path / "2026_Seeding.xlsx",
    source_path / "2026_Seeding.xlsx",
]
seed_file_path = next((p for p in seed_file_candidates if p.exists()), None)

if "dual_pipeline_outputs" not in globals():
    raise RuntimeError("dual_pipeline_outputs is unavailable. Run the club-share export cells first.")

if seed_file_path is None:
    print("No 2026 seeding Excel found for alternate-team seed enrichment.")
else:
    def load_seed_assignments(sheet_name: str) -> pd.DataFrame:
        slots = pd.read_excel(seed_file_path, sheet_name=sheet_name)

        required_cols = {"Season", "Seed", "TeamID"}
        missing_cols = required_cols - set(slots.columns)
        if missing_cols:
            raise KeyError(f"Missing required columns in {sheet_name}: {sorted(missing_cols)}")

        if "AlternateTeamID" not in slots.columns:
            slots["AlternateTeamID"] = np.nan

        slots = slots[["Season", "Seed", "TeamID", "AlternateTeamID"]].copy()
        slots["Season"] = pd.to_numeric(slots["Season"], errors="coerce")
        slots["TeamID"] = pd.to_numeric(slots["TeamID"], errors="coerce")
        slots["AlternateTeamID"] = pd.to_numeric(slots["AlternateTeamID"], errors="coerce")
        slots["Seed"] = slots["Seed"].astype(str).str.strip()

        slots = slots.dropna(subset=["Season", "Seed", "TeamID"]).copy()
        slots = slots[slots["Season"] == season_seed_fix].copy()
        slots = slots[slots["Seed"].str.match(official_seed_pattern, na=False)].copy()
        slots["Season"] = slots["Season"].astype(int)
        slots["TeamID"] = slots["TeamID"].astype(int)
        slots = slots.drop_duplicates(subset=["Season", "Seed"], keep="last")

        primary = slots[["Season", "Seed", "TeamID"]].copy()
        primary["SeedSlotRole"] = "primary"

        alternate = (
            slots[["Season", "Seed", "AlternateTeamID"]]
            .rename(columns={"AlternateTeamID": "TeamID"})
            .dropna(subset=["TeamID"])
            .copy()
        )
        if not alternate.empty:
            alternate["TeamID"] = alternate["TeamID"].astype(int)
        alternate["SeedSlotRole"] = "alternate"

        assignments = pd.concat([primary, alternate], ignore_index=True)
        assignments = assignments.drop_duplicates(subset=["Season", "TeamID", "Seed"], keep="last").copy()

        assignments["SeedNumBase"] = pd.to_numeric(
            assignments["Seed"].str.extract(r"(\d{2})", expand=False),
            errors="coerce",
        )
        assignments["SeedNum"] = assignments["SeedNumBase"].copy()
        assignments["SeedRegion"] = assignments["Seed"].str[0]
        assignments["SeedPlayInSuffix"] = assignments["Seed"].str.extract(r"([a-z])$", expand=False).fillna("").str.lower()
        assignments["SeedPlayInFlag"] = assignments["SeedPlayInSuffix"].ne("").astype(int)

        # Alternate-team rows should be treated as play-in candidates for downstream filtering.
        alt_mask = assignments["SeedSlotRole"].eq("alternate")
        assignments.loc[alt_mask, "SeedPlayInFlag"] = 1
        assignments.loc[alt_mask & assignments["SeedPlayInSuffix"].eq(""), "SeedPlayInSuffix"] = "b"

        return assignments


    for t_type in TOURNAMENT_TYPES:
        assignments = load_seed_assignments(seed_sheet_by_tournament[t_type])
        if assignments.empty:
            print(f"[{t_type}] No 2026 seed assignments found for enrichment.")
            continue

        team_agg_out = dual_pipeline_outputs[t_type]["team_aggregates"].copy()
        team_agg_out["Season"] = pd.to_numeric(team_agg_out["Season"], errors="coerce")
        team_agg_out["TeamID"] = pd.to_numeric(team_agg_out["TeamID"], errors="coerce")

        for col in ["Seed", "SeedNum", "SeedNumBase", "SeedRegion", "SeedPlayInSuffix", "SeedPlayInFlag"]:
            if col not in team_agg_out.columns:
                team_agg_out[col] = np.nan

        team_agg_out = team_agg_out.merge(
            assignments[
                [
                    "Season",
                    "TeamID",
                    "Seed",
                    "SeedNum",
                    "SeedNumBase",
                    "SeedRegion",
                    "SeedPlayInSuffix",
                    "SeedPlayInFlag",
                    "SeedSlotRole",
                ]
            ],
            on=["Season", "TeamID"],
            how="left",
            suffixes=("", "_slot"),
        )

        affected_mask = (
            team_agg_out["Season"].eq(season_seed_fix)
            & team_agg_out["SeedSlotRole"].eq("alternate")
        )

        # Fill from slot assignments when existing values are missing.
        for col in ["Seed", "SeedNum", "SeedNumBase", "SeedRegion", "SeedPlayInSuffix", "SeedPlayInFlag"]:
            slot_col = f"{col}_slot"
            if slot_col in team_agg_out.columns:
                team_agg_out[col] = team_agg_out[col].combine_first(team_agg_out[slot_col])

        # Ensure alternates explicitly carry numeric seeds for downstream logic.
        team_agg_out.loc[affected_mask, "SeedNum"] = pd.to_numeric(
            team_agg_out.loc[affected_mask, "SeedNum"],
            errors="coerce",
        )
        team_agg_out.loc[affected_mask, "SeedNumBase"] = pd.to_numeric(
            team_agg_out.loc[affected_mask, "SeedNumBase"],
            errors="coerce",
        )
        team_agg_out.loc[affected_mask, "SeedPlayInFlag"] = 1

        drop_cols = [
            "Seed_slot",
            "SeedNum_slot",
            "SeedNumBase_slot",
            "SeedRegion_slot",
            "SeedPlayInSuffix_slot",
            "SeedPlayInFlag_slot",
            "SeedSlotRole",
        ]
        team_agg_out = team_agg_out.drop(columns=[c for c in drop_cols if c in team_agg_out.columns])

        updated_rows = int(affected_mask.sum())
        updated_seednums = int(team_agg_out.loc[affected_mask, "SeedNum"].notna().sum())

        dual_pipeline_outputs[t_type]["team_aggregates"] = team_agg_out
        agg_path = dual_pipeline_outputs[t_type]["team_aggregates_path"]
        team_agg_out.to_csv(agg_path, index=False)

        print(f"[{t_type}] Alternate-team seed enrichment saved: {agg_path}")
        print(f"[{t_type}] Alternate rows in 2026: {updated_rows:,} | with SeedNum populated: {updated_seednums:,}")

    # Keep men default handle aligned with downstream cells.
    if "M" in dual_pipeline_outputs:
        team_agg_clean = dual_pipeline_outputs["M"]["team_aggregates"].copy()

[M] Alternate-team seed enrichment saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\m_team_aggregates_2005_2026.csv
[M] Alternate rows in 2026: 4 | with SeedNum populated: 4
[W] Alternate-team seed enrichment saved: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\club_share\w_team_aggregates_2014_2026.csv
[W] Alternate rows in 2026: 4 | with SeedNum populated: 4


In [15]:
# ==========================================
# EXPORT: FIRST-ROUND MATCHUP CANDIDATES (MENS + WOMENS)
# ==========================================
# Run this after the 68-team rebuild and alternate seed enrichment cells.
# This precomputes first-round candidate matchups so teaching notebooks can stay lightweight.
# It derives slot membership from all-possible modeling exports, then merges full modeling metrics.

from pathlib import Path
import re
import numpy as np
import pandas as pd

ROUND1_PAIRS = [(1, 16), (8, 9), (5, 12), (4, 13), (6, 11), (3, 14), (7, 10), (2, 15)]
PLAY_IN_SEEDS = {11, 16}

REQUIRED_EXPORT_COLS = [
    "DataPrefix",
    "Season",
    "Region",
    "FavoriteSeed",
    "UnderdogSeed",
    "FavoriteSeedCode",
    "UnderdogSeedCode",
    "FavoriteTeamID",
    "UnderdogTeamID",
    "FavoriteTeam",
    "UnderdogTeam",
    "Favorite_Final_Massey",
    "Underdog_Final_Massey",
    "Diff_Final_Massey",
    "FavoriteIsAlternate",
    "UnderdogIsAlternate",
    "HasPlayInPath",
    "InModelingFile",
    "FavoriteMatchesModelTeam1",
    "MatchupLabel",
    "MatchupKey",
]


def _resolve_share_path() -> Path:
    if "share_path" in globals():
        return Path(globals()["share_path"])
    if "club_share_path" in globals():
        return Path(globals()["club_share_path"])
    if "processed_path" in globals():
        return Path(globals()["processed_path"]) / "club_share"
    project_root = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
    return project_root / "data" / "processed" / "club_share"


def _resolve_team_agg_path(prefix: str, share_dir: Path) -> Path:
    candidates = sorted(share_dir.glob(f"{prefix}_team_aggregates_*_2026.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"No team aggregate file found for prefix='{prefix}'. "
            f"Expected pattern: {prefix}_team_aggregates_*_2026.csv in {share_dir}"
        )
    return max(candidates, key=lambda p: len(p.stem))


def _extract_seed_num(seed_val):
    match = re.search(r"(\d{2})", str(seed_val))
    return int(match.group(1)) if match else np.nan


def _validate_round1_pairs(round1_pairs):
    if len(round1_pairs) != 8:
        raise ValueError(f"ROUND1_PAIRS must contain 8 seed pairs, found {len(round1_pairs)}")


def build_first_round_candidates(prefix: str, season: int, share_dir: Path) -> pd.DataFrame:
    _validate_round1_pairs(ROUND1_PAIRS)

    team_path = _resolve_team_agg_path(prefix=prefix, share_dir=share_dir)
    matchups_path = share_dir / f"{prefix}_modeling_matchups_2026_all_possible.csv"

    required_paths = [team_path, matchups_path]
    missing = [str(p) for p in required_paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing export inputs:\n" + "\n".join(missing))

    team_df = pd.read_csv(team_path)
    matchups_df = pd.read_csv(matchups_path)

    required_team_cols = {"Season", "TeamID", "TeamName", "Final_Massey"}
    missing_team_cols = sorted(required_team_cols - set(team_df.columns))
    if missing_team_cols:
        raise RuntimeError(
            f"{prefix} team aggregate file missing required columns: {missing_team_cols}"
        )

    required_matchup_cols = {
        "Season",
        "Team1ID",
        "Team2ID",
        "Team1_TeamName",
        "Team2_TeamName",
        "Team1_Seed",
        "Team2_Seed",
        "Team1_SeedRegion",
        "Team2_SeedRegion",
        "Team1_Field68Role",
        "Team2_Field68Role",
    }
    missing_matchup_cols = sorted(required_matchup_cols - set(matchups_df.columns))
    if missing_matchup_cols:
        raise RuntimeError(
            f"{prefix} modeling matchup file missing required columns: {missing_matchup_cols}. "
            f"Run 68-team rebuild before first-round export."
        )

    season_team_df = team_df[pd.to_numeric(team_df["Season"], errors="coerce") == float(season)].copy()
    if season_team_df.empty:
        available = sorted(pd.to_numeric(team_df.get("Season"), errors="coerce").dropna().astype(int).unique().tolist())
        raise RuntimeError(
            f"{prefix} team aggregate has no rows for season={season}. Available seasons: {available[:10]}"
        )

    season_team_df["TeamID"] = pd.to_numeric(season_team_df["TeamID"], errors="coerce")
    season_team_df["Final_Massey"] = pd.to_numeric(season_team_df["Final_Massey"], errors="coerce")
    season_team_df = season_team_df.dropna(subset=["TeamID"]).copy()
    season_team_df["TeamID"] = season_team_df["TeamID"].astype(int)

    team_lookup = (
        season_team_df.sort_values(["TeamID"]).drop_duplicates(subset=["TeamID"], keep="last")
        [["TeamID", "TeamName", "Final_Massey"]]
        .copy()
    )

    matchups_df = matchups_df[pd.to_numeric(matchups_df["Season"], errors="coerce") == float(season)].copy()
    if matchups_df.empty:
        raise RuntimeError(f"{prefix} modeling matchup export has no rows for season={season}.")

    team1_slots = matchups_df[
        ["Team1ID", "Team1_TeamName", "Team1_Seed", "Team1_SeedRegion", "Team1_Field68Role"]
    ].rename(
        columns={
            "Team1ID": "TeamID",
            "Team1_TeamName": "TeamName",
            "Team1_Seed": "Seed",
            "Team1_SeedRegion": "SeedRegion",
            "Team1_Field68Role": "Field68Role",
        }
    )
    team2_slots = matchups_df[
        ["Team2ID", "Team2_TeamName", "Team2_Seed", "Team2_SeedRegion", "Team2_Field68Role"]
    ].rename(
        columns={
            "Team2ID": "TeamID",
            "Team2_TeamName": "TeamName",
            "Team2_Seed": "Seed",
            "Team2_SeedRegion": "SeedRegion",
            "Team2_Field68Role": "Field68Role",
        }
    )

    slot_df = pd.concat([team1_slots, team2_slots], ignore_index=True)
    slot_df["TeamID"] = pd.to_numeric(slot_df["TeamID"], errors="coerce")
    slot_df = slot_df.dropna(subset=["TeamID", "Seed", "SeedRegion"]).copy()
    slot_df["TeamID"] = slot_df["TeamID"].astype(int)
    slot_df["Seed"] = slot_df["Seed"].astype(str).str.strip()
    slot_df["SeedRegion"] = slot_df["SeedRegion"].astype(str).str.upper().str.strip()
    slot_df["SeedNum"] = slot_df["Seed"].apply(_extract_seed_num)
    slot_df = slot_df.dropna(subset=["SeedNum"]).copy()
    slot_df["SeedNum"] = slot_df["SeedNum"].astype(int)
    slot_df = slot_df[slot_df["SeedRegion"].isin(["W", "X", "Y", "Z"])].copy()

    slot_df = slot_df.merge(team_lookup, on="TeamID", how="left", suffixes=("", "_agg"))
    slot_df["TeamName"] = slot_df["TeamName_agg"].fillna(slot_df["TeamName"])
    slot_df = slot_df.drop(columns=[c for c in ["TeamName_agg"] if c in slot_df.columns])

    role_order = {"primary": 0, "alternate": 1}
    slot_df["Field68Role"] = slot_df["Field68Role"].astype(str).str.lower().str.strip()
    slot_df["_role_order"] = slot_df["Field68Role"].map(role_order).fillna(9)
    slot_df = slot_df.sort_values(["SeedRegion", "SeedNum", "_role_order", "TeamID"])
    slot_df = slot_df.drop_duplicates(subset=["TeamID", "SeedRegion", "SeedNum"], keep="first")

    region_seed_to_teams = {}
    for _, row in slot_df.iterrows():
        region_seed_to_teams.setdefault(row["SeedRegion"], {}).setdefault(int(row["SeedNum"]), []).append(row)

    rows = []
    for region, seed_map in sorted(region_seed_to_teams.items()):
        for favorite_seed, underdog_seed in ROUND1_PAIRS:
            favorites = seed_map.get(favorite_seed, [])
            underdogs = seed_map.get(underdog_seed, [])
            for fav in favorites:
                for dog in underdogs:
                    rows.append(
                        {
                            "DataPrefix": prefix,
                            "Season": int(season),
                            "Region": region,
                            "FavoriteSeed": int(favorite_seed),
                            "UnderdogSeed": int(underdog_seed),
                            "FavoriteSeedCode": str(fav.get("Seed", "")),
                            "UnderdogSeedCode": str(dog.get("Seed", "")),
                            "FavoriteTeamID": int(fav.get("TeamID")),
                            "UnderdogTeamID": int(dog.get("TeamID")),
                            "FavoriteTeam": str(fav.get("TeamName", f"Team {int(fav.get('TeamID'))}")),
                            "UnderdogTeam": str(dog.get("TeamName", f"Team {int(dog.get('TeamID'))}")),
                            "Favorite_Final_Massey": float(fav.get("Final_Massey", np.nan)),
                            "Underdog_Final_Massey": float(dog.get("Final_Massey", np.nan)),
                            "FavoriteIsAlternate": str(fav.get("Field68Role", "")).lower() == "alternate",
                            "UnderdogIsAlternate": str(dog.get("Field68Role", "")).lower() == "alternate",
                            "HasPlayInPath": (favorite_seed in PLAY_IN_SEEDS) or (underdog_seed in PLAY_IN_SEEDS),
                        }
                    )

    out = pd.DataFrame(rows)
    if out.empty:
        raise RuntimeError(f"No first-round candidate rows generated for prefix={prefix}, season={season}.")

    out["FavoriteSeed"] = pd.to_numeric(out["FavoriteSeed"], errors="coerce").astype(int)
    out["UnderdogSeed"] = pd.to_numeric(out["UnderdogSeed"], errors="coerce").astype(int)
    out["FavoriteTeamID"] = pd.to_numeric(out["FavoriteTeamID"], errors="coerce").astype(int)
    out["UnderdogTeamID"] = pd.to_numeric(out["UnderdogTeamID"], errors="coerce").astype(int)
    out["Diff_SeedNum"] = out["UnderdogSeed"] - out["FavoriteSeed"]

    out["Diff_Final_Massey"] = (
        pd.to_numeric(out["Favorite_Final_Massey"], errors="coerce")
        - pd.to_numeric(out["Underdog_Final_Massey"], errors="coerce")
    )
    out["MatchupKey"] = out.apply(
        lambda r: tuple(sorted((int(r["FavoriteTeamID"]), int(r["UnderdogTeamID"])))),
        axis=1,
    )
    out["MatchupLabel"] = (
        "(" + out["FavoriteSeedCode"].astype(str) + ") "
        + out["FavoriteTeam"].astype(str)
        + " vs ("
        + out["UnderdogSeedCode"].astype(str)
        + ") "
        + out["UnderdogTeam"].astype(str)
    )

    modeling_with_key = matchups_df.copy()
    modeling_with_key["Team1ID"] = pd.to_numeric(modeling_with_key.get("Team1ID"), errors="coerce")
    modeling_with_key["Team2ID"] = pd.to_numeric(modeling_with_key.get("Team2ID"), errors="coerce")
    modeling_with_key = modeling_with_key.dropna(subset=["Team1ID", "Team2ID"]).copy()
    modeling_with_key["Team1ID"] = modeling_with_key["Team1ID"].astype(int)
    modeling_with_key["Team2ID"] = modeling_with_key["Team2ID"].astype(int)
    modeling_with_key["MatchupKey"] = modeling_with_key.apply(
        lambda r: tuple(sorted((int(r["Team1ID"]), int(r["Team2ID"])))),
        axis=1,
    )
    modeling_with_key = modeling_with_key.drop_duplicates(subset=["MatchupKey"], keep="first")

    rename_map = {c: f"Model_{c}" for c in modeling_with_key.columns if c != "MatchupKey"}
    modeling_with_key = modeling_with_key.rename(columns=rename_map)

    out = out.merge(modeling_with_key, on="MatchupKey", how="left")
    out["InModelingFile"] = out["Model_Team1ID"].notna()
    out["FavoriteMatchesModelTeam1"] = (
        pd.to_numeric(out["Model_Team1ID"], errors="coerce") == out["FavoriteTeamID"]
    )

    model_cols = sorted([c for c in out.columns if c.startswith("Model_")])
    ordered_cols = [
        "DataPrefix",
        "Season",
        "Region",
        "FavoriteSeed",
        "UnderdogSeed",
        "Diff_SeedNum",
        "FavoriteSeedCode",
        "UnderdogSeedCode",
        "FavoriteTeamID",
        "UnderdogTeamID",
        "FavoriteTeam",
        "UnderdogTeam",
        "Favorite_Final_Massey",
        "Underdog_Final_Massey",
        "Diff_Final_Massey",
        "FavoriteIsAlternate",
        "UnderdogIsAlternate",
        "HasPlayInPath",
        "InModelingFile",
        "FavoriteMatchesModelTeam1",
        "MatchupLabel",
        "MatchupKey",
    ]
    out = out[ordered_cols + model_cols].copy()

    missing_required = [c for c in REQUIRED_EXPORT_COLS if c not in out.columns]
    if missing_required:
        raise RuntimeError(f"First-round export is missing required columns: {missing_required}")

    out = out.sort_values(
        ["Region", "FavoriteSeed", "UnderdogSeed", "FavoriteSeedCode", "UnderdogSeedCode"],
        ascending=[True, True, True, True, True],
    ).reset_index(drop=True)

    return out


export_season = int(globals().get("sim_season", 2026))
share_dir = _resolve_share_path()
export_targets = {
    "m": share_dir / f"m_first_round_matchups_{export_season}_68team_candidates.csv",
    "w": share_dir / f"w_first_round_matchups_{export_season}_68team_candidates.csv",
}

exports_by_prefix = {}
for prefix, out_path in export_targets.items():
    export_df = build_first_round_candidates(prefix=prefix, season=export_season, share_dir=share_dir)
    export_df.to_csv(out_path, index=False)
    exports_by_prefix[prefix] = export_df

    play_in_count = int(export_df["HasPlayInPath"].sum())
    alt_rows = int((export_df["FavoriteIsAlternate"] | export_df["UnderdogIsAlternate"]).sum())
    matched_count = int(export_df["InModelingFile"].sum())
    model_cols_merged = len([c for c in export_df.columns if c.startswith("Model_")])

    region_counts = export_df["Region"].value_counts().sort_index().to_dict()
    seed_pair_counts = (
        export_df.groupby(["FavoriteSeed", "UnderdogSeed"]).size().sort_index().to_dict()
    )
    massey_coverage = float(export_df["Diff_Final_Massey"].notna().mean())

    print(f"[{prefix}] wrote {len(export_df):,} rows to {out_path.name}")
    print(f"[{prefix}] region counts: {region_counts}")
    print(f"[{prefix}] seed-pair counts: {seed_pair_counts}")
    print(
        f"[{prefix}] play-in rows: {play_in_count:,} | alternate-seed rows: {alt_rows:,} | "
        f"rows found in modeling file: {matched_count:,}/{len(export_df):,} | model cols merged: {model_cols_merged:,}"
    )
    print(f"[{prefix}] Diff_Final_Massey non-null share: {massey_coverage:.1%}")

if {"m", "w"}.issubset(exports_by_prefix):
    m_cols = exports_by_prefix["m"].columns.tolist()
    w_cols = exports_by_prefix["w"].columns.tolist()
    if m_cols != w_cols:
        raise RuntimeError("First-round export schema mismatch: mens/womens column order differs.")

print("First-round export complete.")

[m] wrote 36 rows to m_first_round_matchups_2026_68team_candidates.csv
[m] region counts: {'W': 8, 'X': 9, 'Y': 10, 'Z': 9}
[m] seed-pair counts: {(1, 16): 6, (2, 15): 4, (3, 14): 4, (4, 13): 4, (5, 12): 4, (6, 11): 6, (7, 10): 4, (8, 9): 4}
[m] play-in rows: 12 | alternate-seed rows: 4 | rows found in modeling file: 36/36 | model cols merged: 73
[m] Diff_Final_Massey non-null share: 100.0%
[w] wrote 36 rows to w_first_round_matchups_2026_68team_candidates.csv
[w] region counts: {'W': 8, 'X': 9, 'Y': 9, 'Z': 10}
[w] seed-pair counts: {(1, 16): 6, (2, 15): 4, (3, 14): 4, (4, 13): 4, (5, 12): 4, (6, 11): 5, (7, 10): 5, (8, 9): 4}
[w] play-in rows: 11 | alternate-seed rows: 4 | rows found in modeling file: 36/36 | model cols merged: 73
[w] Diff_Final_Massey non-null share: 100.0%
First-round export complete.


In [16]:
# ------------------------------------------------------------------------
# QA: 2026 field-size and all-possible export integrity checks
# ------------------------------------------------------------------------
import numpy as np
import pandas as pd

season_qa = 2026
official_seed_regex = r"^[WXYZ]\d{2}[ab]?$"
seed_sheet_by_tournament = {
    "M": "MNCAATourneySeeds",
    "W": "WNCAATourneySeeds",
}
seed_file_candidates = [
    club_share_path / "2026_Seeding.xlsx",
    source_path / "2026_Seeding.xlsx",
]
seed_file_path = next((p for p in seed_file_candidates if p.exists()), None)

if "club_share_path" not in globals():
    club_share_path = processed_path / "club_share"

if seed_file_path is None:
    raise FileNotFoundError("2026_Seeding.xlsx is required for field-size QA checks.")


def load_seed_slots_qa(sheet_name: str) -> pd.DataFrame:
    slots = pd.read_excel(seed_file_path, sheet_name=sheet_name)
    if "AlternateTeamID" not in slots.columns:
        slots["AlternateTeamID"] = np.nan

    slots = slots[["Season", "Seed", "TeamID", "AlternateTeamID"]].copy()
    slots["Season"] = pd.to_numeric(slots["Season"], errors="coerce")
    slots["TeamID"] = pd.to_numeric(slots["TeamID"], errors="coerce")
    slots["AlternateTeamID"] = pd.to_numeric(slots["AlternateTeamID"], errors="coerce")
    slots["Seed"] = slots["Seed"].astype(str).str.strip()
    slots = slots.dropna(subset=["Season", "Seed", "TeamID"]).copy()
    slots = slots[(slots["Season"] == season_qa) & slots["Seed"].str.match(official_seed_regex, na=False)].copy()
    slots["Season"] = slots["Season"].astype(int)
    slots["TeamID"] = slots["TeamID"].astype(int)
    return slots.drop_duplicates(subset=["Season", "Seed"], keep="last")


def build_field_68_qa(seed_slots: pd.DataFrame) -> pd.DataFrame:
    primary = seed_slots[["Season", "Seed", "TeamID"]].copy()
    primary["SeedSlotRole"] = "primary"

    alternate = (
        seed_slots[["Season", "Seed", "AlternateTeamID"]]
        .rename(columns={"AlternateTeamID": "TeamID"})
        .dropna(subset=["TeamID"])
        .copy()
    )
    if not alternate.empty:
        alternate["TeamID"] = alternate["TeamID"].astype(int)
    alternate["SeedSlotRole"] = "alternate"

    field = pd.concat([primary, alternate], ignore_index=True)
    field = field.drop_duplicates(subset=["Season", "Seed", "TeamID"]).copy()
    return field


qa_rows = []
for t_type in TOURNAMENT_TYPES:
    qa_matchup_path = club_share_path / CLUB_SHARE_EXPORT_NAMES[t_type]["all_possible_2026"]

    if "dual_pipeline_outputs" in globals() and t_type in dual_pipeline_outputs:
        qa_matchups = dual_pipeline_outputs[t_type]["all_possible_2026"].copy()
    else:
        if not qa_matchup_path.exists():
            raise FileNotFoundError(f"Could not find {t_type} 2026 matchup export: {qa_matchup_path}")
        qa_matchups = pd.read_csv(qa_matchup_path)

    seed_slots = load_seed_slots_qa(seed_sheet_by_tournament[t_type])
    field_68 = build_field_68_qa(seed_slots)

    seeded_team_count = int(field_68["TeamID"].nunique())
    expected_team_count = int(64 + seed_slots["AlternateTeamID"].notna().sum())
    expected_all_pairs = int(seeded_team_count * (seeded_team_count - 1) // 2)

    qa_matchups["Season"] = pd.to_numeric(qa_matchups["Season"], errors="coerce")
    qa_matchups_2026 = qa_matchups[qa_matchups["Season"] == season_qa].copy()
    actual_all_pairs = int(len(qa_matchups_2026))

    if {"Team1_SeedRegion", "Team2_SeedRegion"}.issubset(qa_matchups_2026.columns):
        same_region_mask = qa_matchups_2026["Team1_SeedRegion"].astype(str) == qa_matchups_2026["Team2_SeedRegion"].astype(str)
        within_region_actual = int(same_region_mask.sum())
        cross_region_actual = int((~same_region_mask).sum())
    else:
        within_region_actual = np.nan
        cross_region_actual = np.nan

    region_counts = field_68.assign(SeedRegion=lambda d: d["Seed"].astype(str).str[0]).groupby("SeedRegion")["TeamID"].nunique().sort_index()
    within_region_expected = int(sum(n * (n - 1) // 2 for n in region_counts.tolist()))
    cross_region_expected = int(expected_all_pairs - within_region_expected)

    qa_rows.extend([
        {"Tournament": t_type, "check": "expected_field_size", "value": expected_team_count},
        {"Tournament": t_type, "check": "actual_field_size", "value": seeded_team_count},
        {"Tournament": t_type, "check": "expected_all_pairs_nC2", "value": expected_all_pairs},
        {"Tournament": t_type, "check": "actual_all_pairs", "value": actual_all_pairs},
        {"Tournament": t_type, "check": "within_region_expected", "value": within_region_expected},
        {"Tournament": t_type, "check": "within_region_actual", "value": within_region_actual},
        {"Tournament": t_type, "check": "cross_region_expected", "value": cross_region_expected},
        {"Tournament": t_type, "check": "cross_region_actual", "value": cross_region_actual},
    ])

    print(f"[{t_type}] QA season: {season_qa}")
    print(f"[{t_type}] Expected field size from seeding file: {expected_team_count}")
    print(f"[{t_type}] Actual unique field teams: {seeded_team_count}")
    print(f"[{t_type}] Teams by region:")
    print(region_counts.to_string())

    if seeded_team_count != expected_team_count:
        raise AssertionError(
            f"[{t_type}] Expected field size {expected_team_count}, got {seeded_team_count}."
        )

    if expected_all_pairs != actual_all_pairs:
        raise AssertionError(
            f"[{t_type}] Mismatch in all-pairs count for {season_qa}: expected {expected_all_pairs}, got {actual_all_pairs}."
        )

    if pd.notna(within_region_actual) and within_region_expected != within_region_actual:
        raise AssertionError(
            f"[{t_type}] Mismatch in within-region pairs for {season_qa}: expected {within_region_expected}, got {within_region_actual}."
        )

    if pd.notna(cross_region_actual) and cross_region_expected != cross_region_actual:
        raise AssertionError(
            f"[{t_type}] Mismatch in cross-region pairs for {season_qa}: expected {cross_region_expected}, got {cross_region_actual}."
        )

summary_df = pd.DataFrame(qa_rows)
display(summary_df)
print("QA passed: 2026 field-size and all-possible matchup export counts are internally consistent.")

[M] QA season: 2026
[M] Expected field size from seeding file: 68
[M] Actual unique field teams: 68
[M] Teams by region:
SeedRegion
W    16
X    17
Y    18
Z    17
[W] QA season: 2026
[W] Expected field size from seeding file: 68
[W] Actual unique field teams: 68
[W] Teams by region:
SeedRegion
W    16
X    17
Y    17
Z    18


,Tournament,check,value
0,M,expected_field_size,68
1,M,actual_field_size,68
2,M,expected_all_pairs_nC2,2278
3,M,actual_all_pairs,2278
4,M,within_region_expected,545
5,M,within_region_actual,545
6,M,cross_region_expected,1733
7,M,cross_region_actual,1733
8,W,expected_field_size,68
9,W,actual_field_size,68


QA passed: 2026 field-size and all-possible matchup export counts are internally consistent.


In [17]:

# -----------------------------------------------------------------
# Diagnostic: Three-Factors composite — validate calibrated weights
# weights_three_factors and compute_composite_weights are set in Cell 12.
# This cell is diagnostics-only; it does NOT produce new features.
# -----------------------------------------------------------------
import numpy as np
import pandas as pd

THREE_FACTOR_COLS = ["Diff_net_efg", "Diff_net_tov", "Diff_net_reb"]

_valid_tf = [c for c in THREE_FACTOR_COLS if c in model_df.columns]
if _valid_tf:
    print("Correlation matrix (three factors — final model_df):")
    display(model_df[_valid_tf].corr())

# Re-fit on final model_df for validation AUC (compare to calibration-pass AUC)
_result_tf_final = compute_composite_weights(model_df, THREE_FACTOR_COLS)
print(f"\nThree-factors AUC (final model_df validation): {_result_tf_final['auc']:.4f}")

print("\nCalibrated three-factors weights (applied in Cell 12 Pass 2):")
_tf_labels = ["net_efg", "net_tov", "net_reb"]
for _lbl, _w_cal, _w_fin in zip(_tf_labels, weights_three_factors, _result_tf_final["weights"]):
    print(f"  {_lbl}: calibrated={_w_cal:+.4f}  |  final-model={_w_fin:+.4f}")


Correlation matrix (three factors — final model_df):


,Diff_net_efg,Diff_net_tov,Diff_net_reb
Diff_net_efg,1.000000,-0.209642,0.053380
Diff_net_tov,-0.209642,1.000000,-0.320582
Diff_net_reb,0.053380,-0.320582,1.000000



Three-factors AUC (final model_df validation): 0.6422

Calibrated three-factors weights (applied in Cell 12 Pass 2):
  net_efg: calibrated=+0.3596  |  final-model=+0.3127
  net_tov: calibrated=+0.3368  |  final-model=+0.3816
  net_reb: calibrated=+0.3036  |  final-model=+0.3057


In [18]:

# -----------------------------------------------------------------
# Diagnostic: Playmaking-Defense composite — validate calibrated weights
# weights_playmaking and compute_composite_weights are set in Cell 12.
# This cell is diagnostics-only; it does NOT produce new features.
# -----------------------------------------------------------------
import numpy as np
import pandas as pd

PLAYMAKING_COLS = ["Diff_net_ast_rate", "Diff_net_stl_pct", "Diff_net_blk_pct"]

_valid_pd = [c for c in PLAYMAKING_COLS if c in model_df.columns]
if _valid_pd:
    print("Correlation matrix (playmaking-defense — final model_df):")
    display(model_df[_valid_pd].corr())

# Re-fit on final model_df for validation AUC
_result_pd_final = compute_composite_weights(model_df, PLAYMAKING_COLS)
print(f"\nPlaymaking-defense AUC (final model_df validation): {_result_pd_final['auc']:.4f}")

print("\nCalibrated playmaking-defense weights (applied in Cell 12 Pass 2):")
_pd_labels = ["net_ast_rate", "net_stl_pct", "net_blk_pct"]
for _lbl, _w_cal, _w_fin in zip(_pd_labels, weights_playmaking, _result_pd_final["weights"]):
    print(f"  {_lbl}: calibrated={_w_cal:+.4f}  |  final-model={_w_fin:+.4f}")


Correlation matrix (playmaking-defense — final model_df):


,Diff_net_ast_rate,Diff_net_stl_pct,Diff_net_blk_pct
Diff_net_ast_rate,1.000000,-0.263662,-0.097974
Diff_net_stl_pct,-0.263662,1.000000,0.128655
Diff_net_blk_pct,-0.097974,0.128655,1.000000



Playmaking-defense AUC (final model_df validation): 0.5937

Calibrated playmaking-defense weights (applied in Cell 12 Pass 2):
  net_ast_rate: calibrated=+0.2973  |  final-model=+0.2630
  net_stl_pct: calibrated=+0.3168  |  final-model=+0.4156
  net_blk_pct: calibrated=+0.3859  |  final-model=+0.3213


In [19]:

# -----------------------------------------------------------------
# Diagnostic: adj_net_rating + avg_three_score + avg_ft_score group
# These remain STANDALONE features in team_features; this cell does NOT
# create a third composite. Shown here for reference only.
# -----------------------------------------------------------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np

factor_cols = [
    "Diff_sos_adj_net_rating",
    "Diff_avg_three_score",
    "Diff_avg_ft_score",
]

# Correlation check
_valid_f3 = [c for c in factor_cols if c in model_df.columns]
if _valid_f3:
    correlation_matrix = model_df[_valid_f3].corr()
    print("Correlation matrix of factors:")
    display(correlation_matrix)

# Fit on final model_df for reference AUC
analysis_df = model_df.dropna(subset=factor_cols + ["Team1Win"]).copy()

X = analysis_df[factor_cols]
y = analysis_df["Team1Win"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

clf = LogisticRegression(fit_intercept=False, random_state=42)
clf.fit(X_scaled, y)

raw_coefficients = clf.coef_[0]

y_pred_proba = clf.predict_proba(X_scaled)[:, 1]
auc_score = roc_auc_score(y, y_pred_proba)
print(f"Reference AUC (adj_net + three_score + ft_score): {auc_score:.3f}")

total_importance = np.sum(np.abs(raw_coefficients))
empirical_weights = raw_coefficients / total_importance

print("--- Reference Weights (diagnostics only — not applied) ---")
for col, weight in zip(factor_cols, empirical_weights):
    print(f"{col}: {weight:.1%}")


Correlation matrix of factors:


,Diff_sos_adj_net_rating,Diff_avg_three_score,Diff_avg_ft_score
Diff_sos_adj_net_rating,1.000000,0.097338,0.008086
Diff_avg_three_score,0.097338,1.000000,-0.141183
Diff_avg_ft_score,0.008086,-0.141183,1.000000


Reference AUC (adj_net + three_score + ft_score): 0.683
--- Reference Weights (diagnostics only — not applied) ---
Diff_sos_adj_net_rating: 79.3%
Diff_avg_three_score: -2.4%
Diff_avg_ft_score: -18.3%


In [20]:
# -----------------------------------------------------------------------------
# Leakage check: compare team-season games_played vs regular-season game counts
# -----------------------------------------------------------------------------
regular_compact_path = source_path / "MRegularSeasonCompactResults.csv"
if not regular_compact_path.exists():
    raise FileNotFoundError(f"Regular season compact file not found: {regular_compact_path}")

if "feature_table" not in globals():
    raise RuntimeError("`feature_table` not found. Run Cell 17 first.")

regular_compact = pd.read_csv(regular_compact_path)
regular_compact = regular_compact[regular_compact["Season"].between(season_start, season_end)].copy()

w_counts = (
    regular_compact.groupby(["Season", "WTeamID"]).size()
    .reset_index(name="w_games")
    .rename(columns={"WTeamID": "TeamID"})
)
l_counts = (
    regular_compact.groupby(["Season", "LTeamID"]).size()
    .reset_index(name="l_games")
    .rename(columns={"LTeamID": "TeamID"})
)

regular_counts = (
    w_counts.merge(l_counts, on=["Season", "TeamID"], how="outer")
    .fillna(0)
)
regular_counts["regular_season_games"] = regular_counts["w_games"] + regular_counts["l_games"]
regular_counts = regular_counts[["Season", "TeamID", "regular_season_games"]]

team_games_leakage = feature_table[["Season", "TeamID", "TeamName", "games_played"]].copy()
team_games_leakage = team_games_leakage.merge(
    regular_counts,
    on=["Season", "TeamID"],
    how="left",
)
team_games_leakage["regular_season_games"] = team_games_leakage["regular_season_games"].fillna(0)
team_games_leakage["games_over_regular"] = team_games_leakage["games_played"] - team_games_leakage["regular_season_games"]
team_games_leakage["games_over_regular_abs"] = team_games_leakage["games_over_regular"].abs()

top_team_overages = team_games_leakage.sort_values(
    ["games_over_regular", "Season", "TeamName"], ascending=[False, True, True]
).head(100)
top_team_overages_out = modeling_diag_path / "top_team_games_over_regular_season.csv"
top_team_overages.to_csv(top_team_overages_out, index=False)

if "model_df" in globals():
    game_level_leakage = model_df[["Season", "DayNum", "Team1ID", "Team2ID", "Team1_TeamName", "Team2_TeamName"]].copy()
    team1_over = team_games_leakage[["Season", "TeamID", "games_over_regular"]].rename(
        columns={"TeamID": "Team1ID", "games_over_regular": "Team1_games_over_regular"}
    )
    team2_over = team_games_leakage[["Season", "TeamID", "games_over_regular"]].rename(
        columns={"TeamID": "Team2ID", "games_over_regular": "Team2_games_over_regular"}
    )
    game_level_leakage = game_level_leakage.merge(team1_over, on=["Season", "Team1ID"], how="left")
    game_level_leakage = game_level_leakage.merge(team2_over, on=["Season", "Team2ID"], how="left")
    game_level_leakage[["Team1_games_over_regular", "Team2_games_over_regular"]] = game_level_leakage[["Team1_games_over_regular", "Team2_games_over_regular"]].fillna(0)
    game_level_leakage["max_games_over_regular"] = game_level_leakage[["Team1_games_over_regular", "Team2_games_over_regular"]].max(axis=1)

    top_matchup_overages = game_level_leakage.sort_values(
        ["max_games_over_regular", "Season", "DayNum"], ascending=[False, True, True]
    ).head(100)
    top_matchup_overages_out = modeling_diag_path / "top_matchup_games_over_regular_season.csv"
    top_matchup_overages.to_csv(top_matchup_overages_out, index=False)

    print(f"Saved matchup-level leakage check: {top_matchup_overages_out}")
    display(top_matchup_overages.head(25))

season_leakage_summary = (
    team_games_leakage.groupby("Season")
    .agg(
        teams=("TeamID", "nunique"),
        avg_games_over_regular=("games_over_regular", "mean"),
        p90_games_over_regular=("games_over_regular", lambda s: s.quantile(0.90)),
        max_games_over_regular=("games_over_regular", "max"),
    )
    .reset_index()
    .sort_values("Season")
)
season_leakage_out = modeling_diag_path / "season_games_over_regular_summary.csv"
season_leakage_summary.to_csv(season_leakage_out, index=False)

print(f"Saved team-level leakage check: {top_team_overages_out}")
print(f"Saved season leakage summary: {season_leakage_out}")
print("Top team overages preview:")
display(top_team_overages.head(25))
print("Season leakage summary:")
display(season_leakage_summary)

Saved matchup-level leakage check: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\top_matchup_games_over_regular_season.csv


,Season,DayNum,Team1ID,Team2ID,Team1_TeamName,Team2_TeamName,Team1_games_over_regular,Team2_games_over_regular,max_games_over_regular
465,2012,136,1307,1253,New Mexico,Long Beach St,7.0,2.0,7.0
490,2012,138,1257,1307,Louisville,New Mexico,0.0,7.0,7.0
526,2013,136,1211,1380,Gonzaga,Southern Univ,1.0,7.0,7.0
1209,2024,137,1155,1307,Clemson,New Mexico,0.0,7.0,7.0
1287,2025,137,1266,1307,Marquette,New Mexico,0.0,7.0,7.0
1304,2025,139,1277,1307,Michigan St,New Mexico,0.0,7.0,7.0
616,2014,137,1307,1390,New Mexico,Stanford,5.0,0.0,5.0
734,2016,136,1425,1344,USC,Providence,0.0,5.0,5.0
761,2016,138,1314,1344,North Carolina,Providence,0.0,5.0,5.0
861,2018,136,1211,1422,Gonzaga,UNC Greensboro,0.0,5.0,5.0


Saved team-level leakage check: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\top_team_games_over_regular_season.csv
Saved season leakage summary: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\season_games_over_regular_summary.csv
Top team overages preview:


,Season,TeamID,TeamName,games_played,regular_season_games,games_over_regular,games_over_regular_abs
2278,2011,1309,New Orleans,21,0.0,21.0,21.0
7700,2026,1380,Southern Univ,37,19.0,18.0,18.0
7662,2026,1341,Prairie View,35,18.0,17.0,17.0
7549,2026,1225,Idaho,35,19.0,16.0,16.0
7618,2026,1295,N Dakota St,37,21.0,16.0,16.0
7623,2026,1300,NC Central,32,16.0,16.0,16.0
7432,2026,1101,Abilene Chr,33,18.0,15.0,15.0
7568,2026,1244,Kennesaw,34,19.0,15.0,15.0
7580,2026,1256,Louisiana Tech,34,19.0,15.0,15.0
7607,2026,1283,Missouri St,34,19.0,15.0,15.0


Season leakage summary:


,Season,teams,avg_games_over_regular,p90_games_over_regular,max_games_over_regular
0,2005,339,-2.994100,0.0,11.0
1,2006,341,-0.378299,2.0,4.0
2,2007,346,-0.150289,2.0,7.0
3,2008,350,0.777143,2.0,6.0
4,2009,353,0.708215,2.0,6.0
5,2010,352,0.315341,2.0,6.0
6,2011,353,1.121813,3.0,21.0
7,2012,349,1.106017,3.0,13.0
8,2013,352,1.000000,2.0,7.0
9,2014,354,1.189266,3.0,11.0


In [21]:
# -------------------------------------------------------------
# Missing team-feature columns report for tournament games only
# Seasons: 2005-2026 (year + team)
# -------------------------------------------------------------

# Load in model df if not already in memory
if "model_df" not in locals():
    model_df = pd.read_csv(model_out_path)

season_start, season_end = 2005, 2026
scope = model_df[model_df["Season"].between(season_start, season_end)].copy()

# Team-side columns that should be populated after joins
exclude_suffixes = {"Season", "TeamID", "TeamName", "SeedRegion"}
team1_feature_cols = [
    c for c in scope.columns
    if c.startswith("Team1_") and c.replace("Team1_", "", 1) not in exclude_suffixes
]

if not team1_feature_cols:
    raise RuntimeError("No Team1 feature columns found to evaluate completeness.")

team2_feature_cols = [c.replace("Team1_", "Team2_", 1) for c in team1_feature_cols if c.replace("Team1_", "Team2_", 1) in scope.columns]

# Normalize both sides into one team-season table
team1_missing = scope[["Season", "Team1ID", "Team1_TeamName"] + team1_feature_cols].copy()
team1_missing = team1_missing.rename(columns={"Team1ID": "TeamID", "Team1_TeamName": "TeamName"})
team1_missing["missing_columns"] = team1_missing[team1_feature_cols].isna().apply(
    lambda row: [col.replace("Team1_", "", 1) for col, is_na in row.items() if is_na], axis=1
)

team2_missing = scope[["Season", "Team2ID", "Team2_TeamName"] + team2_feature_cols].copy()
team2_missing = team2_missing.rename(columns={"Team2ID": "TeamID", "Team2_TeamName": "TeamName"})
team2_missing["missing_columns"] = team2_missing[team2_feature_cols].isna().apply(
    lambda row: [col.replace("Team2_", "", 1) for col, is_na in row.items() if is_na], axis=1
)

combined = pd.concat(
    [
        team1_missing[["Season", "TeamID", "TeamName", "missing_columns"]],
        team2_missing[["Season", "TeamID", "TeamName", "missing_columns"]],
    ],
    ignore_index=True,
)

# Stable team name lookup (prevents NaN TeamName rows from being dropped)
name_lookup = (
    combined[["TeamID", "TeamName"]]
    .dropna(subset=["TeamID"])
    .assign(TeamName=lambda d: d["TeamName"].astype(str).str.strip())
)
name_lookup = name_lookup[name_lookup["TeamName"] != ""]
name_lookup = name_lookup.drop_duplicates("TeamID")

if "teams_df" in globals() and {"TeamID", "TeamName"}.issubset(teams_df.columns):
    canonical_lookup = teams_df[["TeamID", "TeamName"]].drop_duplicates("TeamID")
    name_lookup = canonical_lookup.merge(name_lookup, on="TeamID", how="left", suffixes=("", "_from_model"))
    name_lookup["TeamName"] = name_lookup["TeamName"].combine_first(name_lookup["TeamName_from_model"])
    name_lookup = name_lookup[["TeamID", "TeamName"]]

# Aggregate to unique (Season, Team) with union of missing columns
missing_team_report = (
    combined.groupby(["Season", "TeamID"], as_index=False, dropna=False)["missing_columns"]
    .agg(lambda col_lists: sorted(set(x for sublist in col_lists for x in sublist)))
)
missing_team_report["missing_count"] = missing_team_report["missing_columns"].map(len)
missing_team_report = missing_team_report[missing_team_report["missing_count"] > 0].copy()
missing_team_report = missing_team_report.merge(name_lookup, on="TeamID", how="left")
missing_team_report["TeamName"] = missing_team_report["TeamName"].fillna("<missing team name>")
missing_team_report = missing_team_report.sort_values(["Season", "missing_count", "TeamName"], ascending=[True, False, True])

# Friendly text view + export
missing_team_report["missing_columns_str"] = missing_team_report["missing_columns"].map(lambda cols: ", ".join(cols))

out_path = modeling_diag_path / "tourney_team_missing_features_2005_2026.csv"
missing_team_report[["Season", "TeamID", "TeamName", "missing_count", "missing_columns_str"]].to_csv(out_path, index=False)

print(f"Scope seasons: {season_start}-{season_end}")
print(f"Tournament games in scope: {len(scope):,}")
print(f"Teams with missing columns (year+team): {len(missing_team_report):,}")
print(f"Saved report: {out_path}")

summary_by_season = (
    missing_team_report.groupby("Season", as_index=False)
    .agg(teams_with_missing=("TeamID", "nunique"))
    .sort_values("Season")
)
print("\nTeams with missing columns by season:")
print(summary_by_season.to_string(index=False))

display(missing_team_report[["Season", "TeamID", "TeamName", "missing_count", "missing_columns_str"]].head(100))

Scope seasons: 2005-2026
Tournament games in scope: 1,321
Teams with missing columns (year+team): 1,341
Saved report: x:\My Files\Python\Sports Analytics Projects\NCAA Basketball\March Madness GAM\data\processed\modeling_datasets\diagnostics\tourney_team_missing_features_2005_2026.csv

Teams with missing columns by season:
 Season  teams_with_missing
   2005                  65
   2006                  65
   2007                  65
   2008                  65
   2009                  65
   2010                  65
   2011                  68
   2012                  68
   2013                  68
   2014                  68
   2015                  68
   2016                  68
   2017                  68
   2018                  68
   2019                  68
   2021                  67
   2022                  68
   2023                  68
   2024                  68
   2025                  68


,Season,TeamID,TeamName,missing_count,missing_columns_str
0,2005,1104,Alabama,1,AlternateTeamID
1,2005,1105,Alabama A&M,1,AlternateTeamID
2,2005,1112,Arizona,1,AlternateTeamID
3,2005,1130,Boston College,1,AlternateTeamID
4,2005,1137,Bucknell,1,AlternateTeamID
...,...,...,...,...,...
96,2006,1285,Montana,1,AlternateTeamID
97,2006,1293,Murray St,1,AlternateTeamID
98,2006,1301,NC State,1,AlternateTeamID
99,2006,1305,Nevada,1,AlternateTeamID


# Dual M/W Active Path
The active notebook flow is now a single dual-tournament pipeline: strict split map contracts (`m_team_name_master_map.csv`, `w_team_name_master_map.csv`), dual Massey generation (`m_rating_momentum_features.csv`, `w_rating_momentum_features.csv`), unified feature engineering, parity checks, and prefixed club-share exports for both `M` and `W`.